# Notebook 07 — DINOv2 ViT-S/14 + Five-role 2.5D

**Mục tiêu:** huấn luyện mô hình dự đoán 12 bất thường MRI gối bằng 5 nhóm series giải phẫu: SAG_FLUID, COR_FLUID, AX_FLUID, SAG_NONFLUID và COR_NONFLUID. Notebook này thay thế pipeline DINOv2 single-role cũ; không còn giữ song song model/training loop cũ để tránh nhầm lẫn.

## Cách sử dụng

1. Chọn GPU trong Kaggle Settings. Chạy notebook từ trên xuống để nạp shared labels/cache, audit metadata, tạo supplementary cache (nếu chưa có), tạo Dataset và kiểm tra model.
2. Cell training được **tắt mặc định** (`RUN_FIVE_ROLE_TRAINING = False`). Sau khi Cell model smoke test in `PASS`, đổi thành `True` rồi chạy lại cell training để bắt đầu train.
3. Supplementary cache nằm trong `/kaggle/working/rsna_dinov2_vits14_residual_attention/supplementary_five_role_cache/` và có cơ chế resume. Nếu cache đã tồn tại đúng worklist, preprocessing sẽ bỏ qua các series đã `ok`.
4. Không xóa audit/preprocessing cells: chúng tạo lại được role index và giải thích nguồn dữ liệu. Không cần giữ các cell thử nghiệm cũ của single-role model.

**Split/labels:** giữ nguyên 46 gold train + 12 gold validation; pseudo train chỉ gồm study có ít nhất một target được quan sát. 29 pseudo studies không có nhãn quan sát bị loại khỏi supervised training, không xem chúng là negative.

**Lưu ý:** đầu vào hiện tại giữ 224×224 và normalization tương thích cache cũ. Chưa áp dụng physical 140 mm FOV resampling hoặc laterality mirroring.


## 1. Imports, GPU và môi trường

In [ ]:
import os
import gc
import json
import time
import math
import random
import hashlib
import shutil
import zipfile
import warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, FileLink

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler

from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

warnings.filterwarnings("ignore", category=UserWarning)

print("Python:", __import__("sys").version.split()[0])
print("PyTorch:", torch.__version__)
import torchvision
print("torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU chưa bật. Kaggle Settings → Accelerator → GPU.")

device = torch.device("cuda")
for gpu_i in range(torch.cuda.device_count()):
    prop = torch.cuda.get_device_properties(gpu_i)
    print(f"GPU {gpu_i}: {prop.name} | {prop.total_memory / 1024**3:.1f} GiB")


## 2. Cấu hình thí nghiệm và đường dẫn output

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

ID_COL = "StudyInstanceUID"
TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]

EXPECTED_STUDIES = 4407
EXPECTED_GOLD = 58
EXPECTED_PSEUDO = 4349
MAX_SLICES = 36
IMAGE_SIZE = 224             # DINOv2 input: preserve preprocessing-cache resolution
PATCH_SIZE = 14              # DINOv2 ViT-S/14: 16 x 16 = 256 patch tokens
BACKBONE_DIM = 384           # DINOv2 ViT-S/14 feature dimension
EMBED_DIM = 256              # common study-level representation
STUDY_DEPTH = 2
STUDY_HEADS = 4

# Official DINOv2 backbone
DINO_HUB_REPO = "facebookresearch/dinov2"
DINO_HUB_MODEL = "dinov2_vits14"
DINO_LOCAL_REPO_PATH = ""    # Optional: local clone of facebookresearch/dinov2
FREEZE_EARLY_BLOCKS = 6      # freeze blocks 0..8, fine-tune last 3/12 blocks
SLICE_CHUNK_SIZE = 8         # retained for reference; five-role encoder uses its own chunk size

# Soft center prior: bias attention scores, never crop/mask peripheral regions
USE_CENTER_PRIOR = True
CENTER_PRIOR_STRENGTH = 0.15
CENTER_PRIOR_SIGMA = 0.55

# Same corrected weak-label policy as ViT Residual Attention
GOLD_SAMPLE_WEIGHT = 1.0
PSEUDO_SAMPLE_WEIGHT = 0.10
PSEUDO_TO_GOLD_WEIGHTED_MASS_RATIO = 0.50

INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
OUTPUT_ROOT = WORKING_ROOT / "rsna_dinov2_vits14_residual_attention"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Output:", OUTPUT_ROOT)
print("Targets:", len(TARGETS))
print("Image size:", IMAGE_SIZE, "| patch size:", PATCH_SIZE)
print("DINOv2:", DINO_HUB_MODEL, "| backbone dim:", BACKBONE_DIM)
print("Center prior:", USE_CENTER_PRIOR, "strength:", CENTER_PRIOR_STRENGTH)

# Five-role 2.5D settings
N_ROLE_SLICES = 12
N_ROLE_WINDOWS = N_ROLE_SLICES - 2
FIVE_ROLE_ORDER = [
    "SAG_FLUID", "COR_FLUID", "AX_FLUID", "SAG_NONFLUID", "COR_NONFLUID"
]
FIVE_ROLE_ENCODER_CHUNK = 8


## 3. Helper functions để tìm shared artifacts và preprocessing cache

In [ ]:
SKIP_DIR_NAMES = {
    "train_series", "test_series", "train_images", "test_images",
    ".git", "__pycache__"
}

CACHE_REQUIRED_FILES = {
    "train_slices_uint8.npy", "slice_valid.npy", "plane_ids.npy",
    "fluid_flags.npy", "fat_flags.npy", "slice_positions.npy",
}
REQUIRED_ARTIFACTS = [
    "shared_split.csv",
    "study_labels_training.csv",
    "preprocess_status.csv",
]


def _unique_paths(paths):
    unique = {}
    for path in paths:
        try:
            key = str(Path(path).resolve())
        except OSError:
            key = str(Path(path))
        unique[key] = Path(path)
    return list(unique.values())


def find_named_files(root: Path, filename: str, max_depth: int = 10):
    """Find a file by case-insensitive basename under one root."""
    matches = []
    root = Path(root)
    if not root.exists():
        return matches
    for current, dirs, files in os.walk(root):
        current_path = Path(current)
        try:
            depth = len(current_path.relative_to(root).parts)
        except ValueError:
            depth = max_depth
        dirs[:] = [d for d in dirs if d not in SKIP_DIR_NAMES]
        for name in files:
            if name.lower() == filename.lower():
                matches.append(current_path / name)
        if depth >= max_depth:
            dirs.clear()
    return _unique_paths(matches)


def find_cache_directories(root: Path, max_depth: int = 10):
    """Find directories containing all six TRAIN cache arrays together."""
    found = []
    root = Path(root)
    if not root.exists():
        return found
    for current, dirs, files in os.walk(root):
        current_path = Path(current)
        try:
            depth = len(current_path.relative_to(root).parts)
        except ValueError:
            depth = max_depth
        dirs[:] = [d for d in dirs if d not in SKIP_DIR_NAMES]
        if CACHE_REQUIRED_FILES.issubset(set(files)):
            found.append(current_path)
        if depth >= max_depth:
            dirs.clear()
    return _unique_paths(found)


def find_extracted_artifact_directories(root: Path, max_depth: int = 10):
    """Find extracted shared-artifact folders (the three required CSVs together)."""
    found = []
    root = Path(root)
    if not root.exists():
        return found
    for current, dirs, files in os.walk(root):
        current_path = Path(current)
        try:
            depth = len(current_path.relative_to(root).parts)
        except ValueError:
            depth = max_depth
        dirs[:] = [d for d in dirs if d not in SKIP_DIR_NAMES]
        if set(REQUIRED_ARTIFACTS).issubset(set(files)):
            found.append(current_path)
        if depth >= max_depth:
            dirs.clear()
    return _unique_paths(found)


def common_path_depth(path_a: Path, path_b: Path):
    a = Path(path_a).resolve().parts
    b = Path(path_b).resolve().parts
    n = 0
    for x, y in zip(a, b):
        if x != y:
            break
        n += 1
    return n

print("Kaggle input exists:", INPUT_ROOT.exists())
for p in INPUT_ROOT.iterdir():
    print(" -", p)
print("Kaggle working exists:", WORKING_ROOT.exists())


## 4. Tìm Kaggle input, CNN-LSTM artifacts và cache 224×224

In [ ]:
# CNN-LSTM shared artifacts may be mounted as a ZIP or already extracted.
artifact_zip_input_candidates = find_named_files(
    INPUT_ROOT, "cnn_lstm_model_artifacts.zip"
)
artifact_zip_working_candidates = find_named_files(
    WORKING_ROOT, "cnn_lstm_model_artifacts.zip"
)
# Prefer the mounted Kaggle Input archive; use the persistent working copy only
# when no archive is mounted in /kaggle/input.
artifact_zip_candidates = (
    artifact_zip_input_candidates
    if artifact_zip_input_candidates
    else artifact_zip_working_candidates
)
extracted_artifact_input_candidates = find_extracted_artifact_directories(INPUT_ROOT)
extracted_artifact_working_candidates = find_extracted_artifact_directories(WORKING_ROOT)
extracted_artifact_candidates = (
    extracted_artifact_input_candidates
    if extracted_artifact_input_candidates
    else extracted_artifact_working_candidates
)

# IMPORTANT: File Persistence can leave the six training NPY files in /kaggle/working.
# Search that location as well as /kaggle/input. The six files must be in ONE directory.
working_cache_candidates = find_cache_directories(WORKING_ROOT)
input_cache_candidates = find_cache_directories(INPUT_ROOT)

print("CNN-LSTM artifact ZIP candidates in /kaggle/input:")
for p in artifact_zip_input_candidates:
    print(" -", p)
print("CNN-LSTM artifact ZIP candidates in /kaggle/working:")
for p in artifact_zip_working_candidates:
    print(" -", p)
print("\nExtracted CNN-LSTM artifact directories selected for discovery:")
for p in extracted_artifact_candidates:
    print(" -", p)
print("\nTraining cache candidates in /kaggle/working (File Persistence):")
for p in working_cache_candidates:
    print(" -", p)
print("\nTraining cache candidates in /kaggle/input:")
for p in input_cache_candidates:
    print(" -", p)

if not artifact_zip_candidates and not extracted_artifact_candidates:
    raise FileNotFoundError(
        "Không tìm thấy CNN-LSTM shared artifacts. Cần cnn_lstm_model_artifacts.zip "
        "hoặc thư mục đã giải nén chứa đủ shared_split.csv, "
        "study_labels_training.csv và preprocess_status.csv."
    )

# Prefer the persistent cache in /kaggle/working when present.
# If several cache folders exist in the same location, stop rather than silently
# choosing a potentially stale cache. Set CACHE_DIR_OVERRIDE to one listed path.
CACHE_DIR_OVERRIDE = ""  # Optional: paste an exact cache directory path here.
if CACHE_DIR_OVERRIDE:
    CACHE_DIR = Path(CACHE_DIR_OVERRIDE)
    if not CACHE_REQUIRED_FILES.issubset({p.name for p in CACHE_DIR.glob("*.npy")}):
        raise FileNotFoundError(
            f"CACHE_DIR_OVERRIDE does not contain all six required arrays: {CACHE_DIR}"
        )
elif len(working_cache_candidates) == 1:
    CACHE_DIR = working_cache_candidates[0]
elif len(working_cache_candidates) > 1:
    raise RuntimeError(
        "Tìm thấy nhiều training cache trong /kaggle/working. "
        "Hãy đặt CACHE_DIR_OVERRIDE bằng đúng thư mục cache muốn dùng. Candidates: "
        + "; ".join(str(p) for p in working_cache_candidates)
    )
elif len(input_cache_candidates) == 1:
    CACHE_DIR = input_cache_candidates[0]
elif len(input_cache_candidates) > 1:
    raise RuntimeError(
        "Tìm thấy nhiều training cache trong /kaggle/input. "
        "Hãy đặt CACHE_DIR_OVERRIDE bằng đúng thư mục cache muốn dùng. Candidates: "
        + "; ".join(str(p) for p in input_cache_candidates)
    )
else:
    raise FileNotFoundError(
        "Không tìm thấy training cache đủ 6 file NPY trong cả /kaggle/working "
        "lẫn /kaggle/input. Cần train_slices_uint8.npy cùng 5 metadata arrays "
        "từ cùng một preprocessing run; không dùng test_slices_uint8.npy."
    )

# Prefer a single ZIP if present; otherwise use a single extracted artifact directory.
if len(artifact_zip_candidates) == 1:
    ARTIFACT_ZIP = artifact_zip_candidates[0]
    ARTIFACT_SOURCE_DIR = None
elif len(artifact_zip_candidates) > 1:
    raise RuntimeError(
        "Tìm thấy nhiều cnn_lstm_model_artifacts.zip trong cùng nguồn được ưu tiên. "
        "Hãy giữ lại đúng một bản. Candidates: "
        + "; ".join(str(p) for p in artifact_zip_candidates)
    )
elif len(extracted_artifact_candidates) == 1:
    ARTIFACT_ZIP = None
    ARTIFACT_SOURCE_DIR = extracted_artifact_candidates[0]
elif len(extracted_artifact_candidates) > 1:
    raise RuntimeError(
        "Tìm thấy nhiều thư mục shared artifacts trong cùng nguồn được ưu tiên. "
        "Candidates: " + "; ".join(str(p) for p in extracted_artifact_candidates)
    )
else:
    raise FileNotFoundError("Không tìm thấy CNN-LSTM ZIP hoặc extracted shared artifacts.")

print("\nSelected artifact ZIP:", ARTIFACT_ZIP)
print("Selected extracted artifact dir:", ARTIFACT_SOURCE_DIR)
print("Selected TRAIN cache:", CACHE_DIR)
for name in sorted(CACHE_REQUIRED_FILES):
    p = CACHE_DIR / name
    print(f"  {name}: {p.stat().st_size / 1024**3:.2f} GiB")


## 5. Giải nén shared artifacts

In [ ]:
ARTIFACT_DIR = OUTPUT_ROOT / "cnn_lstm_shared_artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

if ARTIFACT_ZIP is not None:
    with zipfile.ZipFile(ARTIFACT_ZIP, "r") as zf:
        bad_member = zf.testzip()
        if bad_member is not None:
            raise RuntimeError(f"Corrupt member in CNN artifact ZIP: {bad_member}")
        zf.extractall(ARTIFACT_DIR)
elif ARTIFACT_SOURCE_DIR is not None:
    for name in REQUIRED_ARTIFACTS:
        source_file = ARTIFACT_SOURCE_DIR / name
        if not source_file.is_file():
            raise FileNotFoundError(f"Missing shared artifact: {source_file}")
        shutil.copy2(source_file, ARTIFACT_DIR / name)
else:
    raise RuntimeError("No CNN-LSTM artifact source was selected in the discovery cell.")

missing = [name for name in REQUIRED_ARTIFACTS if not (ARTIFACT_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"CNN-LSTM shared artifacts thiếu: {missing}")

for name in REQUIRED_ARTIFACTS:
    p = ARTIFACT_DIR / name
    print(f"{name}: {p.stat().st_size / 1024:.1f} KiB")


## 6. Đọc shared split, labels và preprocessing status

In [ ]:
shared_split = pd.read_csv(ARTIFACT_DIR / "shared_split.csv").fillna("")
label_table = pd.read_csv(ARTIFACT_DIR / "study_labels_training.csv")
preprocess_status = pd.read_csv(ARTIFACT_DIR / "preprocess_status.csv").fillna("")

for frame in [shared_split, label_table, preprocess_status]:
    if ID_COL not in frame.columns:
        raise ValueError(f"Missing column: {ID_COL}")
    frame[ID_COL] = frame[ID_COL].astype(str)

print("shared_split:", shared_split.shape)
print("study_labels_training:", label_table.shape)
print("preprocess_status:", preprocess_status.shape)
display(shared_split["split"].value_counts().rename_axis("split").reset_index(name="studies"))
display(preprocess_status["status"].value_counts(dropna=False).rename_axis("status").reset_index(name="studies"))


## 7. Dựng label/mask matrix và mở preprocessing cache bằng memory-map

In [ ]:
if not label_table[ID_COL].is_unique:
    raise ValueError("Duplicate StudyInstanceUID in label table")
if not shared_split[ID_COL].is_unique:
    raise ValueError("Duplicate StudyInstanceUID in split table")
if len(label_table) != EXPECTED_STUDIES:
    raise ValueError(f"Expected {EXPECTED_STUDIES} labels, got {len(label_table)}")

label_table = label_table.set_index(ID_COL)
shared_split = shared_split.set_index(ID_COL)
preprocess_status = preprocess_status.set_index(ID_COL)

uid_list = label_table.index.astype(str).tolist()
uid_to_idx = {uid: i for i, uid in enumerate(uid_list)}
if set(uid_list) != set(shared_split.index.astype(str)):
    raise ValueError("UID mismatch between labels and shared split")
if set(uid_list) != set(preprocess_status.index.astype(str)):
    raise ValueError("UID mismatch between labels and preprocessing status")

label_values = label_table[TARGETS].apply(pd.to_numeric, errors="coerce")
mask_matrix = label_values.notna().to_numpy(dtype=np.float32)
y_matrix = label_values.fillna(0.0).to_numpy(dtype=np.float32)
if not np.isin(y_matrix, [0.0, 1.0]).all():
    raise ValueError("Labels must be binary after masking missing/uncertain values")

source_names = label_table["label_source"].astype(str).to_numpy()
if set(np.unique(source_names)) - {"gold", "pseudo"}:
    raise ValueError(f"Unexpected label_source values: {np.unique(source_names)}")

# Use explicit policy rather than trusting an old exported sample_weight column.
sample_weights = np.where(
    source_names == "gold", GOLD_SAMPLE_WEIGHT, PSEUDO_SAMPLE_WEIGHT
).astype(np.float32)

split_roles = shared_split.loc[uid_list, "split"].astype(str).to_numpy()
status_values = preprocess_status.loc[uid_list, "status"].astype(str).to_numpy()

n_gold = int((source_names == "gold").sum())
n_pseudo = int((source_names == "pseudo").sum())
print("Gold:", n_gold, "| pseudo:", n_pseudo)
print("Observed target cells:", int(mask_matrix.sum()))
if n_gold != EXPECTED_GOLD or n_pseudo != EXPECTED_PSEUDO:
    raise ValueError(f"Expected gold/pseudo {EXPECTED_GOLD}/{EXPECTED_PSEUDO}, got {n_gold}/{n_pseudo}")


In [ ]:
volume_path = CACHE_DIR / "train_slices_uint8.npy"
valid_path = CACHE_DIR / "slice_valid.npy"
plane_path = CACHE_DIR / "plane_ids.npy"
fluid_path = CACHE_DIR / "fluid_flags.npy"
fat_path = CACHE_DIR / "fat_flags.npy"
position_path = CACHE_DIR / "slice_positions.npy"

volumes = np.load(volume_path, mmap_mode="r")
valid_mm = np.load(valid_path, mmap_mode="r")
plane_mm = np.load(plane_path, mmap_mode="r")
fluid_mm = np.load(fluid_path, mmap_mode="r")
fat_mm = np.load(fat_path, mmap_mode="r")
position_mm = np.load(position_path, mmap_mode="r")

print("Images:", volumes.shape, volumes.dtype)
print("Valid:", valid_mm.shape, valid_mm.dtype)
print("Plane:", plane_mm.shape, plane_mm.dtype)
print("Fluid/Fat/Position:", fluid_mm.shape, fat_mm.shape, position_mm.shape)

expected_shape = (EXPECTED_STUDIES, MAX_SLICES, IMAGE_SIZE, IMAGE_SIZE)
if volumes.shape != expected_shape:
    raise ValueError(f"Expected image cache {expected_shape}, got {volumes.shape}")
if volumes.dtype != np.uint8:
    raise TypeError(f"Expected uint8 image cache, got {volumes.dtype}")
for arr in [valid_mm, plane_mm, fluid_mm, fat_mm, position_mm]:
    if arr.shape != volumes.shape[:2]:
        raise ValueError("A metadata cache has mismatched dimensions")

cache_gib = sum(p.stat().st_size for p in CACHE_DIR.glob("*.npy")) / 1024**3
print(f"Cache total size: {cache_gib:.2f} GiB")


## 8. Giữ nguyên gold split và lọc pseudo studies không có nhãn quan sát

In [ ]:
# ============================================================
# CELL — Gold-only 5-fold multilabel-balanced split
# Replaces the previous fixed 46/12 gold split.
# All 58 gold studies are validation exactly once across 5 folds.
# The 4,320 labelled pseudo studies remain in every fold's training pool.
# ============================================================
import numpy as np
import pandas as pd
import hashlib

cache_ok_ids = {
    uid for uid, status in zip(uid_list, status_values) if str(status) == "ok"
}

# Build the gold and pseudo pools from the authoritative label_source column,
# not from the old shared 46/12 split.
gold_all_ids = sorted(
    uid for uid, source in zip(uid_list, source_names)
    if source == "gold" and uid in cache_ok_ids
    and mask_matrix[uid_to_idx[uid]].sum() > 0
)
pseudo_train_ids = sorted(
    uid for uid, source in zip(uid_list, source_names)
    if source == "pseudo" and uid in cache_ok_ids
    and mask_matrix[uid_to_idx[uid]].sum() > 0
)
label_empty_pseudo_ids = sorted(
    uid for uid, source in zip(uid_list, source_names)
    if source == "pseudo" and uid in cache_ok_ids
    and mask_matrix[uid_to_idx[uid]].sum() == 0
)

if len(gold_all_ids) != EXPECTED_GOLD:
    raise RuntimeError(
        f"Expected {EXPECTED_GOLD} cached labelled gold studies, got {len(gold_all_ids)}."
    )
if len(pseudo_train_ids) != EXPECTED_PSEUDO - 29:
    raise RuntimeError(
        f"Expected 4,320 labelled pseudo studies, got {len(pseudo_train_ids)}. "
        "Recheck source labels/cache status before creating folds."
    )
if len(label_empty_pseudo_ids) != 29:
    raise RuntimeError(
        f"Expected 29 label-empty pseudo studies, got {len(label_empty_pseudo_ids)}."
    )
if set(gold_all_ids) & set(pseudo_train_ids):
    raise RuntimeError("Gold/pseudo pools overlap.")

CV_N_SPLITS = 5
CV_SPLIT_SEED = int(SEED)


def make_multilabel_balanced_folds(y, masks, n_splits=5, seed=42,
                                   n_restarts=12, max_swaps=100):
    """Create fixed-size folds by balancing positive AND negative observed cells.

    This is a deterministic multi-start pair-swap optimizer. It avoids an extra
    package dependency and handles masked/unknown labels explicitly.
    """
    y = np.asarray(y, dtype=np.float64)
    observed = np.asarray(masks, dtype=np.float64) > 0.5
    if y.ndim != 2 or observed.shape != y.shape:
        raise ValueError("y and masks must be matching [studies, targets] matrices.")
    n = len(y)
    if n < n_splits:
        raise ValueError(f"Need at least {n_splits} gold studies, got {n}.")

    fold_sizes = np.full(n_splits, n // n_splits, dtype=np.int64)
    fold_sizes[:n % n_splits] += 1

    # Treat each observed target-positive and target-negative outcome as a
    # separate stratification feature; unknown labels contribute to neither.
    positive = observed & (y >= 0.5)
    negative = observed & (y < 0.5)
    indicators = np.concatenate([positive, negative], axis=1).astype(np.float64)
    if (indicators.sum(axis=1) == 0).any():
        raise ValueError("A gold study has no observed labels and cannot be stratified.")

    rng = np.random.default_rng(seed)
    expected = (fold_sizes / float(n))[:, None] * indicators.sum(axis=0)[None, :]
    denominator = expected + 0.5  # stabilizes rare target/outcome features

    def row_cost(counts, fold_idx):
        return float(np.sum((counts - expected[fold_idx]) ** 2 / denominator[fold_idx]))

    best_assignment = None
    best_cost = float("inf")

    for _restart in range(int(n_restarts)):
        perm = rng.permutation(n)
        assignment = np.empty(n, dtype=np.int64)
        cursor = 0
        for fold_idx, size in enumerate(fold_sizes):
            assignment[perm[cursor:cursor + size]] = fold_idx
            cursor += int(size)

        counts = np.stack([
            indicators[assignment == fold_idx].sum(axis=0)
            for fold_idx in range(n_splits)
        ])
        current_cost = sum(row_cost(counts[f], f) for f in range(n_splits))

        # Greedy best-improvement swaps preserve exact fold sizes.
        for _step in range(int(max_swaps)):
            order = rng.permutation(n)
            best_delta = -1e-12
            best_pair = None
            best_rows = None
            for ai in range(n - 1):
                i = int(order[ai])
                fi = int(assignment[i])
                for bj in range(ai + 1, n):
                    j = int(order[bj])
                    fj = int(assignment[j])
                    if fi == fj:
                        continue
                    diff = indicators[j] - indicators[i]
                    new_i = counts[fi] + diff
                    new_j = counts[fj] - diff
                    delta = (
                        row_cost(new_i, fi) + row_cost(new_j, fj)
                        - row_cost(counts[fi], fi) - row_cost(counts[fj], fj)
                    )
                    if delta < best_delta:
                        best_delta = delta
                        best_pair = (i, j, fi, fj)
                        best_rows = (new_i.copy(), new_j.copy())
            if best_pair is None:
                break
            i, j, fi, fj = best_pair
            assignment[i], assignment[j] = fj, fi
            counts[fi], counts[fj] = best_rows
            current_cost += best_delta

        if current_cost < best_cost:
            best_cost = current_cost
            best_assignment = assignment.copy()

    folds = [np.flatnonzero(best_assignment == f).tolist() for f in range(n_splits)]
    actual_sizes = [len(fold) for fold in folds]
    if actual_sizes != fold_sizes.tolist():
        raise RuntimeError(f"Fold sizes incorrect: {actual_sizes} vs {fold_sizes.tolist()}")
    flattened = [idx for fold in folds for idx in fold]
    if sorted(flattened) != list(range(n)):
        raise RuntimeError("Fold assignment does not cover every gold study exactly once.")
    return folds, best_cost


gold_global_indices = np.asarray([uid_to_idx[uid] for uid in gold_all_ids], dtype=np.int64)
gold_y_for_cv = y_matrix[gold_global_indices]
gold_mask_for_cv = mask_matrix[gold_global_indices]
cv_fold_indices, cv_balance_cost = make_multilabel_balanced_folds(
    gold_y_for_cv,
    gold_mask_for_cv,
    n_splits=CV_N_SPLITS,
    seed=CV_SPLIT_SEED,
)
gold_folds_ids = [
    [gold_all_ids[int(i)] for i in fold]
    for fold in cv_fold_indices
]
gold_fold_by_uid = {
    uid: fold_idx + 1
    for fold_idx, fold_ids in enumerate(gold_folds_ids)
    for uid in fold_ids
}

# Keep fold 1 in the legacy variable names so the existing Cell E, Audit G and
# smoke-test cell can run unchanged. Cell K will subsequently train all 5 folds.
val_study_ids = sorted(gold_folds_ids[0])
gold_val_ids = set(val_study_ids)
gold_train_ids = sorted(set(gold_all_ids) - gold_val_ids)

if len(set(gold_fold_by_uid)) != EXPECTED_GOLD:
    raise RuntimeError("Not every gold study received exactly one validation fold.")
if any(uid not in cache_ok_ids for fold_ids in gold_folds_ids for uid in fold_ids):
    raise RuntimeError("A validation fold contains a study without a valid image cache.")

FIVE_ROLE_CV_ROOT = OUTPUT_ROOT / "five_role_2p5d_v2_cv5"
FIVE_ROLE_CV_ROOT.mkdir(parents=True, exist_ok=True)
cv_assignment_df = pd.DataFrame({
    "StudyInstanceUID": gold_all_ids,
    "fold": [gold_fold_by_uid[uid] for uid in gold_all_ids],
})
cv_assignment_df.to_csv(FIVE_ROLE_CV_ROOT / "gold_5fold_assignments.csv", index=False)

fold_summary_rows = []
for fold_idx, fold_ids in enumerate(gold_folds_ids):
    idx = np.asarray([uid_to_idx[uid] for uid in fold_ids], dtype=np.int64)
    row = {"fold": fold_idx + 1, "n_gold_validation": len(fold_ids)}
    for target_idx, target in enumerate(TARGETS):
        valid = mask_matrix[idx, target_idx] > 0
        labels = y_matrix[idx[valid], target_idx]
        row[f"{target}_positive"] = int((labels >= 0.5).sum())
        row[f"{target}_observed"] = int(len(labels))
    fold_summary_rows.append(row)
cv_fold_distribution_df = pd.DataFrame(fold_summary_rows)

print("========== GOLD 5-FOLD CV SPLIT ==========")
print("Gold studies:", len(gold_all_ids), "| folds:", CV_N_SPLITS)
print("Fold sizes:", [len(x) for x in gold_folds_ids])
print("Labelled pseudo training studies per fold:", len(pseudo_train_ids))
print("Excluded label-empty pseudo studies:", len(label_empty_pseudo_ids))
print("Multilabel balance objective:", round(float(cv_balance_cost), 4))
display(cv_fold_distribution_df)
print("Fold assignments saved:", FIVE_ROLE_CV_ROOT / "gold_5fold_assignments.csv")


## 9. Audit A — Chuẩn hóa series metadata và định nghĩa 5 roles

In [ ]:
# ============================================================
# AUDIT A — Locate train_series.csv and define the five roles
# ============================================================
from pathlib import Path

DEFAULT_COMP_ROOT = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
COMP_ROOT_OVERRIDE = ""  # Optional: set exact competition root if the auto-detection fails.

if COMP_ROOT_OVERRIDE:
    COMP_ROOT = Path(COMP_ROOT_OVERRIDE)
else:
    COMP_ROOT = DEFAULT_COMP_ROOT
    if not (COMP_ROOT / "train_series.csv").is_file():
        csv_candidates = find_named_files(INPUT_ROOT, "train_series.csv")
        if len(csv_candidates) != 1:
            raise FileNotFoundError(
                "Cannot uniquely locate train_series.csv. Set COMP_ROOT_OVERRIDE. "
                f"Candidates: {[str(x) for x in csv_candidates[:10]]}"
            )
        COMP_ROOT = csv_candidates[0].parent

TRAIN_SERIES_CSV = COMP_ROOT / "train_series.csv"
if not TRAIN_SERIES_CSV.is_file():
    raise FileNotFoundError(TRAIN_SERIES_CSV)

# DICOM directory name used by the competition input. Detect from known layouts.
TRAIN_SERIES_DIR_OVERRIDE = ""  # Optional exact directory containing StudyUID/SeriesUID folders.
if TRAIN_SERIES_DIR_OVERRIDE:
    TRAIN_SERIES_DIR = Path(TRAIN_SERIES_DIR_OVERRIDE)
else:
    possible_dicom_roots = [
        COMP_ROOT / "train_images",
        COMP_ROOT / "train_series",
        COMP_ROOT / "train_dicom",
        COMP_ROOT / "train",
    ]
    TRAIN_SERIES_DIR = next((p for p in possible_dicom_roots if p.is_dir()), None)
    if TRAIN_SERIES_DIR is None:
        raise FileNotFoundError(
            "Cannot find the DICOM root (expected train_images/train_series/train_dicom). "
            "Set TRAIN_SERIES_DIR_OVERRIDE."
        )

series_meta = pd.read_csv(TRAIN_SERIES_CSV, dtype={
    "StudyInstanceUID": str,
    "SeriesInstanceUID": str,
})
required_cols = {
    "StudyInstanceUID", "SeriesInstanceUID", "Fluid_Sensitive",
    "Fat_Suppression", "Anatomical_Plane",
}
missing_cols = required_cols - set(series_meta.columns)
if missing_cols:
    raise ValueError(f"train_series.csv is missing columns: {sorted(missing_cols)}")

series_meta["StudyInstanceUID"] = series_meta["StudyInstanceUID"].astype(str)
series_meta["SeriesInstanceUID"] = series_meta["SeriesInstanceUID"].astype(str)

def _normalize_plane(value):
    s = str(value).strip().lower()
    if "sag" in s:
        return "sagittal"
    if "cor" in s:
        return "coronal"
    if "axi" in s or "trans" in s or s == "ax":
        return "axial"
    return s

def _normalize_binary_flag(series):
    text = series.astype(str).str.strip().str.lower()
    mapped = text.map({
        "true": 1, "false": 0, "yes": 1, "no": 0,
        "y": 1, "n": 0, "1.0": 1, "0.0": 0,
        "1": 1, "0": 0,
    })
    numeric = pd.to_numeric(series, errors="coerce")
    return mapped.where(mapped.notna(), numeric)

series_meta["plane_norm"] = series_meta["Anatomical_Plane"].map(_normalize_plane)
series_meta["fluid_num"] = _normalize_binary_flag(series_meta["Fluid_Sensitive"])
series_meta["fat_num"] = _normalize_binary_flag(series_meta["Fat_Suppression"])

ROLE_SPECS = {
    "SAG_FLUID":    ("sagittal", 0, 1, "role_SAG_FLUID"),
    "COR_FLUID":    ("coronal",  1, 1, "role_COR_FLUID"),
    "AX_FLUID":     ("axial",    2, 1, "role_AX_FLUID"),
    "SAG_NONFLUID": ("sagittal", 0, 0, "role_SAG_NONFLUID"),
    "COR_NONFLUID": ("coronal",  1, 0, "role_COR_NONFLUID"),
}
for role, (plane, _, fluid_flag, col) in ROLE_SPECS.items():
    series_meta[col] = (
        series_meta["plane_norm"].eq(plane)
        & series_meta["fluid_num"].eq(fluid_flag)
    )

if series_meta["SeriesInstanceUID"].duplicated().any():
    raise ValueError("SeriesInstanceUID must be unique in train_series.csv")

role_summary_rows = []
for role, (_, _, _, col) in ROLE_SPECS.items():
    candidate = series_meta[series_meta[col]]
    role_summary_rows.append({
        "role": role,
        "n_series": int(len(candidate)),
        "n_studies": int(candidate["StudyInstanceUID"].nunique()),
    })
role_summary_df = pd.DataFrame(role_summary_rows)
study_role_counts = series_meta.groupby("StudyInstanceUID")[[x[3] for x in ROLE_SPECS.values()]].any().sum(axis=1)

print("========== SERIES ROLE AUDIT ==========")
print("Competition root:", COMP_ROOT)
print("DICOM root:", TRAIN_SERIES_DIR)
print("train_series.csv:", series_meta.shape)
print("Studies:", series_meta["StudyInstanceUID"].nunique())
display(role_summary_df)
print("Candidate roles per study:")
display(study_role_counts.value_counts().sort_index().rename_axis("n_roles").reset_index(name="n_studies"))
if int(study_role_counts.min()) < 3:
    raise RuntimeError("At least one study has fewer than three candidate roles; inspect metadata mapping.")


## 10. Audit B — Spot-check DICOM headers và geometry

In [ ]:
# ============================================================
# AUDIT B — Lightweight DICOM header spot-check (not pixel decoding)
# ============================================================
import pydicom

RUN_DICOM_HEADER_AUDIT = True
if RUN_DICOM_HEADER_AUDIT:
    # Select up to three studies that have all five candidate roles.
    role_cols = [spec[3] for spec in ROLE_SPECS.values()]
    study_role_matrix = series_meta.groupby("StudyInstanceUID")[role_cols].any()
    complete_uids = study_role_matrix.index[study_role_matrix.all(axis=1)].astype(str).tolist()
    sample_uids = complete_uids[:3]
    if len(sample_uids) < 1:
        raise RuntimeError("No studies with all five candidate roles for DICOM spot-check.")

    header_rows = []
    tags = [
        "StudyInstanceUID", "SeriesInstanceUID", "SOPInstanceUID",
        "SeriesDescription", "ProtocolName", "SequenceName", "ImageType",
        "Rows", "Columns", "PixelSpacing", "ImageOrientationPatient",
        "ImagePositionPatient", "SliceThickness", "SpacingBetweenSlices",
        "InstanceNumber", "FrameOfReferenceUID",
    ]
    for uid in sample_uids:
        group = series_meta[series_meta["StudyInstanceUID"].eq(uid)]
        for role, (_, _, _, col) in ROLE_SPECS.items():
            candidates = group[group[col]].copy()
            if candidates.empty:
                continue
            if role.endswith("NONFLUID"):
                candidates["_fat_rank"] = candidates["fat_num"].map({0.0: 0, 1.0: 2}).fillna(1)
                candidates = candidates.sort_values(["_fat_rank", "SeriesInstanceUID"])
            else:
                candidates = candidates.sort_values("SeriesInstanceUID")
            rec = candidates.iloc[0]
            folder = TRAIN_SERIES_DIR / uid / str(rec["SeriesInstanceUID"])
            files = sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() == ".dcm")
            if not files:
                files = sorted(p for p in folder.iterdir() if p.is_file() and not p.name.startswith("."))
            if not files:
                raise FileNotFoundError(f"No DICOM files in {folder}")
            sample_paths = sorted({files[0], files[len(files)//2], files[-1]})
            for path in sample_paths:
                ds = pydicom.dcmread(str(path), stop_before_pixels=True, force=True, specific_tags=tags)
                row = {"StudyInstanceUID": uid, "role": role, "SeriesInstanceUID": str(rec["SeriesInstanceUID"]),
                       "filename": path.name}
                for tag in tags:
                    value = getattr(ds, tag, None)
                    row[tag] = str(value) if value is not None else ""
                header_rows.append(row)
    header_audit_df = pd.DataFrame(header_rows)
    print("========== DICOM HEADER SPOT-CHECK ==========")
    print("Headers read:", len(header_audit_df), "| studies:", len(sample_uids))
    display(header_audit_df[["role", "SeriesDescription", "Rows", "Columns", "PixelSpacing",
                             "ImageOrientationPatient", "ImagePositionPatient", "SliceThickness"]].head(20))
    geometry_cols = ["PixelSpacing", "ImageOrientationPatient", "ImagePositionPatient"]
    print("Geometry completeness:")
    display(pd.DataFrame({c: [int(header_audit_df[c].astype(str).str.len().gt(0).sum()), len(header_audit_df)]
                          for c in geometry_cols}, index=["present", "total"]).T)
else:
    print("DICOM header spot-check skipped (previously audited).")


## 11. Cell C — Tạo five-role index và supplementary worklist

In [ ]:
# ============================================================
# CELL C — Build a five-role index and supplementary worklist
# Run after cache, labels, and the two audit cells have been loaded.
# ============================================================
import json
import numpy as np
import pandas as pd
from pathlib import Path

SUPP_ROOT = OUTPUT_ROOT / "supplementary_five_role_cache"
SUPP_ROOT.mkdir(parents=True, exist_ok=True)
SUPP_IMAGE_SIZE = IMAGE_SIZE       # 224; match existing cache
SUPP_SLICES_PER_SERIES = 12
SUPP_MIN_DECODED = 4
SUPP_SAVE_EVERY = 10

ROLE_SPECS = {
    "SAG_FLUID":    ("sagittal", 0, 1, "role_SAG_FLUID"),
    "COR_FLUID":    ("coronal",  1, 1, "role_COR_FLUID"),
    "AX_FLUID":     ("axial",    2, 1, "role_AX_FLUID"),
    "SAG_NONFLUID": ("sagittal", 0, 0, "role_SAG_NONFLUID"),
    "COR_NONFLUID": ("coronal",  1, 0, "role_COR_NONFLUID"),
}

# Exact series-to-cache mapping exported by CNN-LSTM.
selected_path = ARTIFACT_DIR / "selected_series_index.csv"
if not selected_path.is_file():
    matches = list(OUTPUT_ROOT.rglob("selected_series_index.csv"))
    if len(matches) != 1:
        raise FileNotFoundError(
            "Need selected_series_index.csv from cnn_lstm_model_artifacts.zip."
        )
    selected_path = matches[0]

selected_df = pd.read_csv(selected_path, dtype={
    "StudyInstanceUID": str, "SeriesInstanceUID": str
})
selected_df["StudyInstanceUID"] = selected_df["StudyInstanceUID"].astype(str)
selected_df["SeriesInstanceUID"] = selected_df["SeriesInstanceUID"].astype(str)
selected_df["plane_norm"] = selected_df["Anatomical_Plane"].astype(str).str.strip().str.lower()
selected_df["fluid_num"] = pd.to_numeric(selected_df["Fluid_Sensitive"], errors="coerce").fillna(-1).astype(int)
selected_df["fat_num"] = pd.to_numeric(selected_df["Fat_Suppression"], errors="coerce").fillna(-1).astype(int)
selected_lookup = {
    (r.StudyInstanceUID, r.plane_norm): {
        "series_uid": r.SeriesInstanceUID, "fluid": int(r.fluid_num), "fat": int(r.fat_num)
    }
    for r in selected_df.itertuples(index=False)
}

series_meta["StudyInstanceUID"] = series_meta["StudyInstanceUID"].astype(str)
series_meta["SeriesInstanceUID"] = series_meta["SeriesInstanceUID"].astype(str)
series_groups = {str(uid): g for uid, g in series_meta.groupby("StudyInstanceUID", sort=False)}

role_rows, task_rows = [], []
for uid in uid_list:
    row_idx = int(uid_to_idx[uid])
    g = series_groups.get(uid, pd.DataFrame())
    v = np.asarray(valid_mm[row_idx]).astype(bool)
    p = np.asarray(plane_mm[row_idx]).astype(int)
    f = np.asarray(fluid_mm[row_idx]).astype(int)

    for role, (plane_name, plane_id, fluid_flag, role_col) in ROLE_SPECS.items():
        candidates = g[g[role_col].fillna(False).astype(bool)].copy() if len(g) and role_col in g else pd.DataFrame()
        cache_slots = np.flatnonzero(v & (p == plane_id) & (f == fluid_flag)).astype(int).tolist()
        old_selected = selected_lookup.get((uid, plane_name))
        old_matches = old_selected is not None and old_selected["fluid"] == fluid_flag

        # Reuse exact original slices when this role is already in CNN-LSTM cache.
        if len(cache_slots) >= SUPP_MIN_DECODED and old_matches:
            role_rows.append({
                "StudyInstanceUID": uid, "row_idx": row_idx, "role": role,
                "plane_id": plane_id, "fluid_flag": fluid_flag,
                "fat_flag": old_selected["fat"],
                "SeriesInstanceUID": old_selected["series_uid"],
                "source": "main_cache", "cache_slice_indices": json.dumps(cache_slots),
                "supp_row_idx": -1, "available": True, "status": "ok",
            })
            continue

        if candidates.empty:
            role_rows.append({
                "StudyInstanceUID": uid, "row_idx": row_idx, "role": role,
                "plane_id": plane_id, "fluid_flag": fluid_flag, "fat_flag": -1,
                "SeriesInstanceUID": "", "source": "missing",
                "cache_slice_indices": "[]", "supp_row_idx": -1,
                "available": False, "status": "no_candidate_series",
            })
            continue

        # Reuse the old selected SeriesUID if it has the desired role but its
        # cached pixels are insufficient. Otherwise pick non-fat-suppressed first.
        chosen = None
        if old_matches:
            exact = candidates[candidates["SeriesInstanceUID"].eq(old_selected["series_uid"])]
            if len(exact):
                chosen = exact.iloc[0]
        if chosen is None:
            fat = pd.to_numeric(candidates["Fat_Suppression"], errors="coerce")
            candidates["_fat_rank"] = np.select([fat.eq(0), fat.eq(1)], [0, 2], default=1)
            chosen = candidates.sort_values(
                ["_fat_rank", "SeriesInstanceUID"], kind="mergesort"
            ).iloc[0]

        fat_value = pd.to_numeric(pd.Series([chosen["Fat_Suppression"]]), errors="coerce").iloc[0]
        fat_flag = int(fat_value) if pd.notna(fat_value) and fat_value in (0, 1) else -1
        task = {
            "StudyInstanceUID": uid, "row_idx": row_idx, "role": role,
            "plane_id": plane_id, "fluid_flag": fluid_flag,
            "fat_flag": fat_flag, "SeriesInstanceUID": str(chosen["SeriesInstanceUID"]),
            "source": "supplementary", "cache_slice_indices": "[]",
            "supp_row_idx": len(task_rows), "available": True, "status": "pending",
        }
        task_rows.append(task)
        role_rows.append(task.copy())

role_index_df = pd.DataFrame(role_rows)
supp_tasks_df = pd.DataFrame(task_rows)
if role_index_df.duplicated(["StudyInstanceUID", "role"]).any():
    raise ValueError("Duplicate study-role rows in five-role index.")

role_index_df.to_csv(SUPP_ROOT / "five_role_index_pending.csv", index=False)
supp_tasks_df.to_csv(SUPP_ROOT / "supplementary_tasks.csv", index=False)

print("========== FIVE-ROLE SOURCE MAP ==========")
display(role_index_df.groupby(["role", "source"]).size().rename("n_studies").reset_index())
print("\nSupplementary series to decode:", len(supp_tasks_df))
display(supp_tasks_df.groupby("role").size().rename("n_series_to_decode").reset_index())
print("\nExpected non-fluid gaps: SAG 4,009 + COR 3,247 = 7,256 series,")
print("minus/plus any cache decode failures discovered by the exact per-role check above.")
print("Worklist:", SUPP_ROOT / "supplementary_tasks.csv")


## 12. Cell D — Supplementary DICOM preprocessing có thể resume

In [ ]:
# ============================================================
# CELL D — Resumable DICOM preprocessing for missing roles
# Writes uint8 images + normalized/physical positions + slice identity.
# ============================================================
import os, re, json, time, shutil
import numpy as np
import pandas as pd
import pydicom
import cv2
from pydicom.pixels import apply_modality_lut
from pathlib import Path

SUPP_ROOT = OUTPUT_ROOT / "supplementary_five_role_cache"
tasks = pd.read_csv(
    SUPP_ROOT / "supplementary_tasks.csv",
    dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str},
).sort_values("supp_row_idx").reset_index(drop=True)
N, K, S = len(tasks), SUPP_SLICES_PER_SERIES, SUPP_IMAGE_SIZE
if N == 0:
    print("No supplementary series to decode.")
else:
    need_gib = N * K * S * S / 1024**3
    image_file = SUPP_ROOT / "supplementary_role_slices_uint8.npy"
    create_gib = 0.0 if image_file.exists() else need_gib
    free_gib = shutil.disk_usage(str(SUPP_ROOT)).free / 1024**3
    print(f"Series to decode: {N:,} | total image cache: {need_gib:.2f} GiB | new disk needed: {create_gib:.2f} GiB | free: {free_gib:.2f} GiB")
    if free_gib < create_gib * 1.15:
        raise OSError("Insufficient Kaggle Working disk; stop before decoding.")

    paths = {
        "images": SUPP_ROOT / "supplementary_role_slices_uint8.npy",
        "valid": SUPP_ROOT / "supplementary_role_valid.npy",
        "pos": SUPP_ROOT / "supplementary_role_position_norm.npy",
        "posmm": SUPP_ROOT / "supplementary_role_position_mm.npy",
        "status": SUPP_ROOT / "supplementary_task_status.csv",
        "slices": SUPP_ROOT / "supplementary_slice_manifest.csv",
    }
    shapes = {
        "images": ((N, K, S, S), np.uint8, 0),
        "valid": ((N, K), np.uint8, 0),
        "pos": ((N, K), np.float32, 0.0),
        "posmm": ((N, K), np.float32, np.nan),
    }
    arrays = {}
    for name, (shape, dtype, fill) in shapes.items():
        if paths[name].exists():
            arr = np.load(paths[name], mmap_mode="r+")
            if arr.shape != shape or arr.dtype != np.dtype(dtype):
                raise ValueError(f"Existing {name} cache has wrong shape/dtype: {arr.shape}/{arr.dtype}")
        else:
            arr = np.lib.format.open_memmap(paths[name], mode="w+", dtype=dtype, shape=shape)
            arr[...] = fill
            arr.flush()
        arrays[name] = arr

    # Status file makes the long decode resumable.
    keys = ["supp_row_idx", "StudyInstanceUID", "role", "SeriesInstanceUID"]
    if paths["status"].exists():
        status_df = pd.read_csv(paths["status"], dtype={
            "StudyInstanceUID": str, "SeriesInstanceUID": str
        })
        if len(status_df) != N or any(
            not status_df[c].astype(str).equals(tasks[c].astype(str)) for c in keys
        ):
            raise ValueError("Existing status file belongs to a different worklist.")
    else:
        status_df = tasks.copy()
        status_df["status"] = "pending"
        status_df["error"] = ""
        for c in ["n_files", "n_decoded"]:
            status_df[c] = 0
        for c in ["sort_mode", "SeriesDescription", "ProtocolName", "SequenceName",
                  "Laterality", "Rows", "Columns", "PixelSpacing",
                  "ImageOrientationPatient", "FrameOfReferenceUID",
                  "SliceThickness", "SpacingBetweenSlices"]:
            status_df[c] = ""
        status_df.to_csv(paths["status"], index=False)

    def _sf(x, default=np.nan):
        try: return float(x)
        except Exception: return default

    def _json_seq(x):
        if x is None: return ""
        try: return json.dumps([float(v) for v in x])
        except Exception: return str(x)

    def _natural(name):
        return [int(p) if p.isdigit() else p.lower() for p in re.split(r"(\d+)", name)]

    TAGS = [
        "ImagePositionPatient", "ImageOrientationPatient", "InstanceNumber", "SOPInstanceUID",
        "PixelSpacing", "Rows", "Columns", "SeriesDescription", "ProtocolName", "SequenceName",
        "Laterality", "ImageLaterality", "FrameOfReferenceUID", "SliceThickness",
        "SpacingBetweenSlices", "PhotometricInterpretation", "RescaleSlope", "RescaleIntercept",
    ]

    def _read_headers(folder):
        files = sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() == ".dcm")
        if not files:
            files = sorted(p for p in folder.iterdir() if p.is_file() and not p.name.startswith("."))
        rows, first_normal, series_info = [], None, {}
        for path in files:
            try:
                ds = pydicom.dcmread(str(path), stop_before_pixels=True, force=True, specific_tags=TAGS)
                if not series_info:
                    for key in ["SeriesDescription", "ProtocolName", "SequenceName", "Laterality",
                                "ImageLaterality", "Rows", "Columns", "PixelSpacing",
                                "ImageOrientationPatient", "FrameOfReferenceUID", "SliceThickness",
                                "SpacingBetweenSlices"]:
                        value = getattr(ds, key, None)
                        series_info[key] = _json_seq(value) if key in ("PixelSpacing", "ImageOrientationPatient") else (str(value) if value is not None else "")
                iop, ipp = getattr(ds, "ImageOrientationPatient", None), getattr(ds, "ImagePositionPatient", None)
                pos, inst = np.nan, _sf(getattr(ds, "InstanceNumber", np.nan))
                if iop is not None and ipp is not None and len(iop) >= 6 and len(ipp) >= 3:
                    normal = np.cross(np.asarray(iop[:3], float), np.asarray(iop[3:6], float))
                    norm = np.linalg.norm(normal)
                    if norm > 0:
                        normal /= norm
                        if first_normal is None: first_normal = normal
                        pos = float(np.dot(np.asarray(ipp[:3], float), first_normal))
                rows.append({"path": path, "filename": path.name, "position_mm": pos,
                             "instance": inst, "sop_uid": str(getattr(ds, "SOPInstanceUID", ""))})
            except Exception as exc:
                rows.append({"path": path, "filename": path.name, "position_mm": np.nan,
                             "instance": np.nan, "sop_uid": "", "header_error": str(exc)})
        if not rows: return [], "no_dicom_files", {}
        n = len(rows)
        npos = sum(np.isfinite(r["position_mm"]) for r in rows)
        ninst = sum(np.isfinite(r["instance"]) for r in rows)
        if npos >= max(2, int(.7*n)):
            rows.sort(key=lambda r: (np.inf if not np.isfinite(r["position_mm"]) else r["position_mm"], r["filename"]))
            mode = "ImagePositionPatient_projected"
        elif ninst >= max(2, int(.7*n)):
            rows.sort(key=lambda r: (np.inf if not np.isfinite(r["instance"]) else r["instance"], r["filename"]))
            mode = "InstanceNumber"
        else:
            rows.sort(key=lambda r: _natural(r["filename"]))
            mode = "filename_fallback"
        for i, r in enumerate(rows): r["sort_index"] = i
        return rows, mode, series_info

    def _decode(path):
        ds = pydicom.dcmread(str(path), force=True)
        x = np.asarray(ds.pixel_array)
        if x.ndim == 3:
            if x.shape[-1] in (3, 4):
                x = x[..., :3].astype(np.float32)
                x = .299*x[..., 0] + .587*x[..., 1] + .114*x[..., 2]
            else:
                x = x[x.shape[0] // 2]
        x = np.squeeze(x)
        if x.ndim != 2: raise ValueError(f"Expected 2D image, got {x.shape}")
        x = x.astype(np.float32)
        try: x = np.asarray(apply_modality_lut(x, ds), dtype=np.float32)
        except Exception:
            x = x * _sf(getattr(ds, "RescaleSlope", 1), 1) + _sf(getattr(ds, "RescaleIntercept", 0), 0)
        if str(getattr(ds, "PhotometricInterpretation", "MONOCHROME2")).upper() == "MONOCHROME1":
            lo, hi = np.nanmin(x), np.nanmax(x)
            x = hi + lo - x
        x[~np.isfinite(x)] = 0
        return x, ds

    def _process(task):
        uid, series_uid = str(task["StudyInstanceUID"]), str(task["SeriesInstanceUID"])
        folder = TRAIN_SERIES_DIR / uid / series_uid
        if not folder.exists(): return None, [], {}, f"missing_series_directory:{folder}"
        headers, sort_mode, info = _read_headers(folder)
        if not headers: return None, [], info, sort_mode
        n = len(headers)
        initial = np.unique(np.linspace(0, n-1, min(K, n)).round().astype(int)).tolist()
        candidates = initial + [i for i in range(n) if i not in set(initial)]
        imgs, metas, errors = [], [], []
        for hidx in candidates:
            h = headers[hidx]
            try:
                image, ds = _decode(h["path"])
                actual_study = str(getattr(ds, "StudyInstanceUID", ""))
                actual_series = str(getattr(ds, "SeriesInstanceUID", ""))
                if actual_study and actual_study != uid:
                    raise ValueError(f"StudyInstanceUID mismatch: {actual_study} != {uid}")
                if actual_series and actual_series != series_uid:
                    raise ValueError(f"SeriesInstanceUID mismatch: {actual_series} != {series_uid}")
                imgs.append(image)
                metas.append({
                    "supp_row_idx": int(task["supp_row_idx"]),
                    "StudyInstanceUID": uid, "role": str(task["role"]),
                    "SeriesInstanceUID": series_uid, "slice_slot": len(imgs)-1,
                    "source_filename": h["filename"], "SOPInstanceUID": str(getattr(ds, "SOPInstanceUID", h["sop_uid"])),
                    "sort_index": h["sort_index"], "position_norm": h["sort_index"]/max(n-1,1),
                    "position_mm": h["position_mm"], "InstanceNumber": str(getattr(ds, "InstanceNumber", "")),
                    "ImagePositionPatient": _json_seq(getattr(ds, "ImagePositionPatient", None)),
                    "ImageOrientationPatient": _json_seq(getattr(ds, "ImageOrientationPatient", None)),
                    "PixelSpacing": _json_seq(getattr(ds, "PixelSpacing", None)),
                    "Rows": int(getattr(ds, "Rows", image.shape[0])),
                    "Columns": int(getattr(ds, "Columns", image.shape[1])),
                    "Laterality": str(getattr(ds, "ImageLaterality", getattr(ds, "Laterality", ""))),
                    "FrameOfReferenceUID": str(getattr(ds, "FrameOfReferenceUID", "")),
                    "SliceThickness": _sf(getattr(ds, "SliceThickness", np.nan)),
                    "SpacingBetweenSlices": _sf(getattr(ds, "SpacingBetweenSlices", np.nan)),
                })
                if len(imgs) >= K: break
            except Exception as exc:
                errors.append(f"{h['filename']}:{type(exc).__name__}:{exc}")
        if len(imgs) < SUPP_MIN_DECODED:
            return None, [], info, "too_few_decoded_slices:" + " | ".join(errors[:2])
        order = np.argsort([m["sort_index"] for m in metas])
        imgs = [imgs[i] for i in order]
        metas = [metas[i] for i in order]
        # Match CNN-LSTM cache normalization: percentile 1.0 / 99.5 over sampled slices.
        vals = np.concatenate([x.reshape(-1) for x in imgs])
        lo, hi = np.percentile(vals, [1.0, 99.5])
        if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
            lo, hi = float(vals.min()), float(vals.max())
        if hi <= lo: hi = lo + 1.0
        out = np.zeros((K, S, S), np.uint8)
        valid = np.zeros(K, np.uint8)
        pos = np.zeros(K, np.float32)
        posmm = np.full(K, np.nan, np.float32)
        for j, (x, meta) in enumerate(zip(imgs, metas)):
            x = np.clip((np.clip(x, lo, hi)-lo)/(hi-lo), 0, 1)
            x = (x*255).round().astype(np.uint8)
            out[j] = cv2.resize(x, (S, S), interpolation=cv2.INTER_AREA)
            valid[j] = 1
            pos[j] = float(meta["position_norm"])
            posmm[j] = float(meta["position_mm"])
            meta["slice_slot"] = j
        info.update({
            "n_files": n, "n_decoded": len(imgs), "sort_mode": sort_mode,
            "decode_errors": " | ".join(errors[:3]),
        })
        return (out, valid, pos, posmm), metas, info, "ok"

    slice_buffer, started, done = [], time.time(), 0
    for i in range(N):
        if str(status_df.at[i, "status"]) == "ok" and int(arrays["valid"][i].sum()) >= SUPP_MIN_DECODED:
            continue
        task = tasks.iloc[i].to_dict()
        try:
            result, slice_rows, info, status = _process(task)
        except Exception as exc:
            result, slice_rows, info = None, [], {}
            status = f"exception:{type(exc).__name__}:{exc}"
        if status == "ok" and result is not None:
            images, valid, pos, posmm = result
            arrays["images"][i], arrays["valid"][i] = images, valid
            arrays["pos"][i], arrays["posmm"][i] = pos, posmm
            slice_buffer.extend(slice_rows)
        else:
            arrays["images"][i] = 0
            arrays["valid"][i] = 0
            arrays["pos"][i] = 0
            arrays["posmm"][i] = np.nan
        status_df.at[i, "status"] = status
        status_df.at[i, "error"] = "" if status == "ok" else status
        for col in ["n_files", "n_decoded", "sort_mode", "SeriesDescription", "ProtocolName",
                    "SequenceName", "Laterality", "Rows", "Columns", "PixelSpacing",
                    "ImageOrientationPatient", "FrameOfReferenceUID", "SliceThickness",
                    "SpacingBetweenSlices"]:
            status_df.at[i, col] = info.get(col, "")
        done += 1
        if done % SUPP_SAVE_EVERY == 0:
            for a in arrays.values(): a.flush()
            if slice_buffer:
                pd.DataFrame(slice_buffer).to_csv(
                    paths["slices"], mode="a", header=not paths["slices"].exists(), index=False
                )
                slice_buffer.clear()
            status_df.to_csv(paths["status"], index=False)
            print(
                f"Run processed={done} | OK={int(status_df.status.eq('ok').sum())}/{N} "
                f"| elapsed={(time.time()-started)/60:.1f} min", flush=True
            )

    for a in arrays.values(): a.flush()
    if slice_buffer:
        pd.DataFrame(slice_buffer).to_csv(
            paths["slices"], mode="a", header=not paths["slices"].exists(), index=False
        )
    status_df.to_csv(paths["status"], index=False)

    # Deduplicate if interrupted after appending slice metadata but before status save.
    if paths["slices"].exists():
        sm = pd.read_csv(paths["slices"], low_memory=False)
        sm = sm.drop_duplicates(["supp_row_idx", "slice_slot"], keep="last")
        sm = sm.sort_values(["supp_row_idx", "slice_slot"])
        sm.to_csv(paths["slices"], index=False)

    # Final role index links old-cache roles and supplementary roles without copying pixels.
    role_index = pd.read_csv(
        SUPP_ROOT / "five_role_index_pending.csv",
        dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str}
    )
    status_lookup = status_df.set_index("supp_row_idx")["status"].to_dict()
    is_supp = role_index.source.eq("supplementary")
    role_index.loc[is_supp, "status"] = role_index.loc[is_supp, "supp_row_idx"].map(status_lookup)
    role_index.loc[is_supp, "available"] = role_index.loc[is_supp, "status"].eq("ok")
    role_index.to_csv(SUPP_ROOT / "five_role_index.csv", index=False)

    valid_counts = np.asarray(arrays["valid"]).sum(axis=1)
    bad_ok = [i for i in range(N) if status_df.at[i, "status"] == "ok" and valid_counts[i] < SUPP_MIN_DECODED]
    if bad_ok:
        raise RuntimeError(f"{len(bad_ok)} tasks marked OK but have too few valid slices.")

    manifest = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "n_supplementary_series": N,
        "n_success": int(status_df.status.eq("ok").sum()),
        "n_failed": int(status_df.status.ne("ok").sum()),
        "roles": ["SAG_FLUID", "COR_FLUID", "AX_FLUID", "SAG_NONFLUID", "COR_NONFLUID"],
        "image_size": S, "slices_per_series": K,
        "intensity_percentiles": [1.0, 99.5],
        "slice_sort_priority": ["ImagePositionPatient projected onto slice normal", "InstanceNumber", "natural filename"],
        "saved_geometry": ["SOPInstanceUID", "ImagePositionPatient", "ImageOrientationPatient", "PixelSpacing", "SliceThickness", "SpacingBetweenSlices", "FrameOfReferenceUID", "Laterality"],
        "note": "224px full-image resize and 1.0/99.5 percentile normalization match original cache. No 140mm FOV resampling or laterality mirroring is applied yet.",
    }
    (SUPP_ROOT / "supplementary_manifest.json").write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
    )

    print("========== SUPPLEMENTARY PREPROCESSING COMPLETE ==========")
    display(status_df.groupby(["role", "status"]).size().rename("n_series").reset_index())
    print("\nFive-role availability:")
    final_role = pd.read_csv(SUPP_ROOT / "five_role_index.csv")
    display(final_role.groupby(["role", "source", "available"]).size().rename("n_studies").reset_index())
    print("\nMost common DICOM SeriesDescription by role:")
    if "SeriesDescription" in status_df.columns:
        display(
            status_df[status_df["status"].eq("ok")]
            .groupby("role")["SeriesDescription"]
            .value_counts().groupby(level=0).head(12)
            .rename("n_series").reset_index()
        )
    print("\nSaved to:", SUPP_ROOT)


## 13. Cell E — Five-role 2.5D Dataset và DataLoaders

In [ ]:
# ============================================================
# CELL E — Build five-role 2.5D Dataset + DataLoaders
# Run after supplementary preprocessing Cell D has completed.
# Does NOT train the model.
# ============================================================
import json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

FIVE_ROLE_ROOT = OUTPUT_ROOT / "supplementary_five_role_cache"
FIVE_ROLE_INDEX_PATH = FIVE_ROLE_ROOT / "five_role_index.csv"
if not FIVE_ROLE_INDEX_PATH.is_file():
    raise FileNotFoundError(
        f"Không tìm thấy {FIVE_ROLE_INDEX_PATH}. Hãy chạy xong Cell D trước."
    )

FIVE_ROLE_ORDER = [
    "SAG_FLUID", "COR_FLUID", "AX_FLUID", "SAG_NONFLUID", "COR_NONFLUID"
]
N_ROLE_SLICES = 12
N_ROLE_WINDOWS = N_ROLE_SLICES - 2  # 10 overlapping 3-slice windows

role_index_df = pd.read_csv(
    FIVE_ROLE_INDEX_PATH,
    dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str},
)
role_index_df["StudyInstanceUID"] = role_index_df["StudyInstanceUID"].astype(str)
if role_index_df.duplicated(["StudyInstanceUID", "role"]).any():
    raise ValueError("five_role_index.csv có duplicate StudyInstanceUID + role.")
if set(role_index_df["role"].unique()) != set(FIVE_ROLE_ORDER):
    raise ValueError("Role names trong five_role_index.csv không khớp FIVE_ROLE_ORDER.")

# Every labelled study must have exactly five index rows, including explicit missing rows.
expected_pairs = len(uid_list) * len(FIVE_ROLE_ORDER)
if len(role_index_df) != expected_pairs:
    raise ValueError(
        f"Expected {expected_pairs:,} study-role rows, got {len(role_index_df):,}."
    )
if set(role_index_df["StudyInstanceUID"]) != set(uid_list):
    raise ValueError("StudyInstanceUID mismatch between five-role index and label table.")

# The supplementary arrays are opened as memmaps; this does not load ~4 GiB into RAM.
supp_images = np.load(FIVE_ROLE_ROOT / "supplementary_role_slices_uint8.npy", mmap_mode="r")
supp_valid = np.load(FIVE_ROLE_ROOT / "supplementary_role_valid.npy", mmap_mode="r")
supp_pos_norm = np.load(FIVE_ROLE_ROOT / "supplementary_role_position_norm.npy", mmap_mode="r")
supp_status_df = pd.read_csv(
    FIVE_ROLE_ROOT / "supplementary_task_status.csv",
    dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str},
)

if supp_images.ndim != 4 or supp_images.shape[1:] != (12, IMAGE_SIZE, IMAGE_SIZE):
    raise ValueError(f"Unexpected supplementary image shape: {supp_images.shape}")
if supp_images.dtype != np.uint8:
    raise TypeError(f"Supplementary images must be uint8, got {supp_images.dtype}")
if supp_valid.shape != supp_images.shape[:2] or supp_pos_norm.shape != supp_images.shape[:2]:
    raise ValueError("Supplementary valid/position arrays do not match image cache.")
if len(supp_status_df) != len(supp_images):
    raise ValueError("Supplementary status rows do not match supplementary image rows.")

# Use a stable Python lookup. The pixels themselves remain memory-mapped.
role_lookup = {
    (str(r.StudyInstanceUID), str(r.role)): r._asdict()
    for r in role_index_df.itertuples(index=False)
}


def _parse_slice_indices(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    parsed = json.loads(str(value))
    return [int(x) for x in parsed]


class FiveRoleStudyDataset(Dataset):
    """One study -> five roles x ten overlapping 2.5D windows."""

    def __init__(self, study_ids):
        self.study_ids = [str(x) for x in study_ids]
        if len(self.study_ids) != len(set(self.study_ids)):
            raise ValueError("study_ids contains duplicates.")
        unknown = sorted(set(self.study_ids) - set(uid_to_idx))
        if unknown:
            raise KeyError(f"Unknown study IDs (first 3): {unknown[:3]}")
        self.global_indices = np.asarray([uid_to_idx[x] for x in self.study_ids], dtype=np.int64)

    def __len__(self):
        return len(self.study_ids)

    def _load_role_slices(self, uid, global_idx, role):
        rec = role_lookup[(uid, role)]
        source = str(rec["source"])
        out_images = np.zeros((N_ROLE_SLICES, IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8)
        out_valid = np.zeros(N_ROLE_SLICES, dtype=np.uint8)
        out_pos = np.zeros(N_ROLE_SLICES, dtype=np.float32)

        if source == "missing" or not bool(rec["available"]):
            return out_images, out_valid, out_pos

        if source == "main_cache":
            if int(rec["row_idx"]) != int(global_idx):
                raise ValueError(f"row_idx mismatch for {uid}/{role} in main cache.")
            slot_ids = _parse_slice_indices(rec["cache_slice_indices"])
            if not slot_ids:
                raise ValueError(f"No cache_slice_indices for main-cache role {uid}/{role}.")
            if min(slot_ids) < 0 or max(slot_ids) >= volumes.shape[1]:
                raise IndexError(f"Invalid main-cache slice index for {uid}/{role}.")
            slot_ids = np.asarray(slot_ids, dtype=np.int64)
            sl_valid = np.asarray(valid_mm[global_idx, slot_ids]).astype(np.uint8)
            sl_pos = np.asarray(position_mm[global_idx, slot_ids], dtype=np.float32)
            # Sort by the already-normalized position, not by filename or UID.
            order_key = np.where(np.isfinite(sl_pos), sl_pos, np.arange(len(sl_pos)))
            order = np.argsort(order_key, kind="stable")
            slot_ids, sl_valid, sl_pos = slot_ids[order], sl_valid[order], sl_pos[order]
            if len(slot_ids) > N_ROLE_SLICES:
                take = np.linspace(0, len(slot_ids) - 1, N_ROLE_SLICES).round().astype(int)
                slot_ids, sl_valid, sl_pos = slot_ids[take], sl_valid[take], sl_pos[take]
            n = min(len(slot_ids), N_ROLE_SLICES)
            out_images[:n] = np.asarray(volumes[global_idx, slot_ids[:n]], dtype=np.uint8)
            out_valid[:n] = sl_valid[:n]
            out_pos[:n] = np.nan_to_num(sl_pos[:n], nan=0.0, posinf=0.0, neginf=0.0)

        elif source == "supplementary":
            supp_idx = int(rec["supp_row_idx"])
            if not (0 <= supp_idx < len(supp_images)):
                raise IndexError(f"Invalid supp_row_idx for {uid}/{role}: {supp_idx}")
            status_row = supp_status_df.iloc[supp_idx]
            if str(status_row["status"]) != "ok":
                raise RuntimeError(
                    f"Index says {uid}/{role} is available but supplementary status is "
                    f"{status_row['status']}."
                )
            if str(status_row["StudyInstanceUID"]) != uid:
                raise ValueError(f"Supplementary UID mismatch at row {supp_idx}.")
            if str(status_row["role"]) != role:
                raise ValueError(f"Supplementary role mismatch at row {supp_idx}.")
            out_images[:] = np.asarray(supp_images[supp_idx], dtype=np.uint8)
            out_valid[:] = np.asarray(supp_valid[supp_idx], dtype=np.uint8)
            out_pos[:] = np.nan_to_num(
                np.asarray(supp_pos_norm[supp_idx], dtype=np.float32),
                nan=0.0, posinf=0.0, neginf=0.0
            )
        else:
            raise ValueError(f"Unknown role source: {source}")

        # A role with no valid decoded pixels is not a usable role.
        if out_valid.sum() == 0:
            return np.zeros_like(out_images), np.zeros_like(out_valid), np.zeros_like(out_pos)
        return out_images, out_valid, out_pos

    def __getitem__(self, item):
        uid = self.study_ids[item]
        global_idx = int(self.global_indices[item])
        all_windows = np.zeros(
            (len(FIVE_ROLE_ORDER), N_ROLE_WINDOWS, 3, IMAGE_SIZE, IMAGE_SIZE),
            dtype=np.uint8,
        )
        all_window_valid = np.zeros((len(FIVE_ROLE_ORDER), N_ROLE_WINDOWS), dtype=np.uint8)
        all_window_pos = np.zeros((len(FIVE_ROLE_ORDER), N_ROLE_WINDOWS), dtype=np.float32)
        role_available = np.zeros(len(FIVE_ROLE_ORDER), dtype=np.uint8)

        for role_idx, role in enumerate(FIVE_ROLE_ORDER):
            slices, slice_valid, slice_pos = self._load_role_slices(uid, global_idx, role)
            for w in range(N_ROLE_WINDOWS):
                w_valid = slice_valid[w:w + 3]
                # Allow one missing channel, but require at least two real slices.
                if int(w_valid.sum()) >= 2:
                    all_windows[role_idx, w] = slices[w:w + 3]
                    all_window_valid[role_idx, w] = 1
                    all_window_pos[role_idx, w] = float(slice_pos[w:w + 3][w_valid.astype(bool)].mean())
            role_available[role_idx] = int(all_window_valid[role_idx].any())

        return {
            "images": torch.from_numpy(all_windows),
            "window_valid": torch.from_numpy(all_window_valid),
            "window_positions": torch.from_numpy(all_window_pos),
            "role_available": torch.from_numpy(role_available),
            "labels": torch.from_numpy(y_matrix[global_idx].astype(np.float32)),
            "label_mask": torch.from_numpy(mask_matrix[global_idx].astype(np.float32)),
            "sample_weight": torch.tensor(sample_weights[global_idx], dtype=torch.float32),
            "uid": uid,
        }


# Reuse exactly the established 46/12 gold split and existing pseudo training pool.
five_role_train_ids = list(gold_train_ids) + list(pseudo_train_ids)
five_role_val_ids = list(val_study_ids)
if set(five_role_train_ids) & set(five_role_val_ids):
    raise ValueError("Five-role train/validation leakage detected.")

five_role_train_dataset = FiveRoleStudyDataset(five_role_train_ids)
five_role_val_dataset = FiveRoleStudyDataset(five_role_val_ids)
FIVE_ROLE_BATCH_SIZE = 1  # conservative for the initial 2.5D smoke test
FIVE_ROLE_NUM_WORKERS = 0  # robust with Kaggle memmaps; increase only after validation
five_role_train_loader = DataLoader(
    five_role_train_dataset, batch_size=FIVE_ROLE_BATCH_SIZE,
    shuffle=True, num_workers=FIVE_ROLE_NUM_WORKERS, pin_memory=torch.cuda.is_available(),
)
five_role_val_loader = DataLoader(
    five_role_val_dataset, batch_size=FIVE_ROLE_BATCH_SIZE,
    shuffle=False, num_workers=FIVE_ROLE_NUM_WORKERS, pin_memory=torch.cuda.is_available(),
)

print("========== FIVE-ROLE DATASET READY ==========")
print("Role order:", FIVE_ROLE_ORDER)
print("Train studies:", len(five_role_train_dataset), "| validation studies:", len(five_role_val_dataset))
print("Per study: images [5, 10, 3, 224, 224] | window mask [5, 10]")
print("Supplementary memmap:", supp_images.shape, "| dtype:", supp_images.dtype)
print("Final availability by role:")
display(
    role_index_df.assign(available=role_index_df["available"].astype(str).str.lower().eq("true"))
    .groupby(["role", "available"]).size().rename("n_studies").reset_index()
)


## 14. Audit G — Xác nhận train/validation coverage và role availability

In [ ]:
# ============================================================
# AUDIT G — Confirm all studies are assigned or label-empty
# ============================================================
train_ids_5r = list(gold_train_ids) + list(pseudo_train_ids)
val_ids_5r = list(val_study_ids)
assigned_ids = set(train_ids_5r) | set(val_ids_5r)
excluded_ids = sorted(set(uid_list) - assigned_ids)

split_audit_rows = []
for uid in excluded_ids:
    i = uid_to_idx[uid]
    split_audit_rows.append({
        "StudyInstanceUID": uid,
        "label_source": str(source_names[i]),
        "observed_targets": int(mask_matrix[i].sum()),
        "sample_weight": float(sample_weights[i]),
    })
split_audit_df = pd.DataFrame(split_audit_rows)

print("========== SPLIT COVERAGE AUDIT ==========")
print("All indexed studies:", len(uid_list))
print("Gold train:", len(gold_train_ids))
print("Pseudo train with observed labels:", len(pseudo_train_ids))
print("Gold validation:", len(val_ids_5r))
print("Train union:", len(train_ids_5r))
print("Train/validation overlap:", len(set(train_ids_5r) & set(val_ids_5r)))
print("Excluded studies:", len(excluded_ids))
if len(excluded_ids):
    display(split_audit_df)
    if not (split_audit_df["label_source"].eq("pseudo").all()
            and split_audit_df["observed_targets"].eq(0).all()):
        raise RuntimeError("Excluded studies include non-empty labels or gold studies; inspect split.")

if len(gold_train_ids) != 46 or len(pseudo_train_ids) != 4320 or len(val_ids_5r) != 12:
    raise RuntimeError(
        f"Unexpected split counts: gold={len(gold_train_ids)}, pseudo={len(pseudo_train_ids)}, val={len(val_ids_5r)}"
    )
if set(train_ids_5r) & set(val_ids_5r):
    raise RuntimeError("Train/validation leakage detected.")
if any(mask_matrix[uid_to_idx[u]].sum() == 0 for u in train_ids_5r):
    raise RuntimeError("A label-empty study entered supervised training.")

availability = (
    role_index_df.assign(available=role_index_df["available"].astype(str).str.lower().eq("true"))
    .pivot(index="StudyInstanceUID", columns="role", values="available")
)
available_roles_per_study = availability.sum(axis=1)
print("Available role count per study:")
display(available_roles_per_study.value_counts().sort_index().rename_axis("n_available_roles").reset_index(name="n_studies"))
if int(available_roles_per_study.min()) < 3:
    raise RuntimeError("At least one study has fewer than three usable roles after preprocessing.")
print("PASS — 46 gold train + 4,320 labelled pseudo train + 12 gold validation; 29 label-empty pseudo excluded.")


## 15. Load official pretrained DINOv2 ViT-S/14

In [ ]:
def load_official_dinov2_backbone():
    """Load official DINOv2 ViT-S/14 with pretrained weights; never random-init."""
    local_repo = Path(DINO_LOCAL_REPO_PATH).expanduser() if DINO_LOCAL_REPO_PATH else None

    if local_repo is not None:
        if not (local_repo / "hubconf.py").is_file():
            raise FileNotFoundError(
                f"DINO_LOCAL_REPO_PATH has no hubconf.py: {local_repo}"
            )
        print("Loading DINOv2 from local repository:", local_repo)
        backbone = torch.hub.load(
            str(local_repo), DINO_HUB_MODEL, source="local", pretrained=True
        )
        source = f"local repo: {local_repo}; pretrained=True"
        return backbone, source

    # Prefer an already-cached official Hub repository when available.
    hub_dir = Path(torch.hub.get_dir())
    cached_repos = sorted(hub_dir.glob("facebookresearch_dinov2_*"))
    cached_repos = [p for p in cached_repos if (p / "hubconf.py").is_file()]

    if cached_repos:
        local_repo = cached_repos[-1]
        print("Using cached official DINOv2 repository:", local_repo)
        backbone = torch.hub.load(
            str(local_repo), DINO_HUB_MODEL, source="local", pretrained=True
        )
        source = f"cached official repo: {local_repo}; pretrained=True"
        return backbone, source

    print("No cached DINOv2 repository found. Loading official PyTorch Hub model...")
    try:
        backbone = torch.hub.load(
            DINO_HUB_REPO,
            DINO_HUB_MODEL,
            pretrained=True,
            trust_repo=True,
        )
    except TypeError:
        # Compatibility with older PyTorch versions without trust_repo argument.
        backbone = torch.hub.load(
            DINO_HUB_REPO, DINO_HUB_MODEL, pretrained=True
        )
    source = f"{DINO_HUB_REPO}:{DINO_HUB_MODEL}; pretrained=True"
    return backbone, source

try:
    PRETRAINED_BACKBONE, PRETRAINED_SOURCE = load_official_dinov2_backbone()
except Exception as exc:
    raise RuntimeError(
        "Không load được pretrained DINOv2. Hãy bật Internet trong Kaggle "
        "cho lần tải đầu tiên hoặc cung cấp DINO_LOCAL_REPO_PATH tới bản clone "
        "chính thức đã có checkpoint pretrained trong torch.hub/checkpoints. "
        "Notebook không cho phép random initialization fallback."
    ) from exc

if not hasattr(PRETRAINED_BACKBONE, "forward_features"):
    raise TypeError("Loaded backbone does not expose DINOv2 forward_features().")
if not hasattr(PRETRAINED_BACKBONE, "blocks") or len(PRETRAINED_BACKBONE.blocks) != 12:
    raise RuntimeError("Expected official DINOv2 ViT-S/14 with exactly 12 blocks.")

print("Pretrained source:", PRETRAINED_SOURCE)
print("Transformer blocks:", len(PRETRAINED_BACKBONE.blocks))
print("Embedding dimension:", BACKBONE_DIM)
print("Parameters (M):", round(sum(p.numel() for p in PRETRAINED_BACKBONE.parameters()) / 1e6, 2))


## 16. Cell H — Gold-prioritized sampler và per-target loss weights

In [ ]:
# ============================================================
# CELL H — Five-fold gold-prioritized samplers and fold DataLoaders
# Uses the 5 folds from Cell 17. No model training happens here.
# ============================================================
import math
import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader, Sampler

GOLD_WEIGHT_5R = 1.0
PSEUDO_WEIGHT_5R = 0.05
PSEUDO_MASS_RATIO_5R = 0.50
FIVE_ROLE_BATCH_SIZE = 1
FIVE_ROLE_GRAD_ACCUM = 4
FIVE_ROLE_NUM_WORKERS = 0


class FiveRoleGoldPrioritizedSampler(Sampler):
    """Use every gold study and a fresh pseudo subset each epoch."""
    def __init__(self, gold_positions, pseudo_positions, n_pseudo, seed=SEED):
        self.gold_positions = np.asarray(gold_positions, dtype=np.int64)
        self.pseudo_positions = np.asarray(pseudo_positions, dtype=np.int64)
        self.n_pseudo = int(n_pseudo)
        self.seed = int(seed)
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        if self.n_pseudo <= len(self.pseudo_positions):
            chosen = rng.choice(self.pseudo_positions, size=self.n_pseudo, replace=False)
        else:
            chosen = rng.choice(self.pseudo_positions, size=self.n_pseudo, replace=True)
        indices = np.concatenate([self.gold_positions, chosen])
        rng.shuffle(indices)
        return iter(indices.tolist())

    def __len__(self):
        return len(self.gold_positions) + self.n_pseudo


def build_five_role_fold_data(fold_idx):
    """Build one fold's train/validation datasets, sampler, loaders and pos_weight."""
    fold_idx = int(fold_idx)
    if not 0 <= fold_idx < CV_N_SPLITS:
        raise IndexError(f"Invalid CV fold index: {fold_idx}")

    fold_val_ids = list(gold_folds_ids[fold_idx])
    fold_val_set = set(fold_val_ids)
    fold_gold_train_ids = sorted(set(gold_all_ids) - fold_val_set)
    fold_train_ids = fold_gold_train_ids + list(pseudo_train_ids)

    if len(fold_gold_train_ids) + len(fold_val_ids) != len(gold_all_ids):
        raise RuntimeError(f"Fold {fold_idx + 1}: gold coverage mismatch.")
    if set(fold_gold_train_ids) & fold_val_set:
        raise RuntimeError(f"Fold {fold_idx + 1}: gold train/validation leakage.")
    if set(fold_train_ids) & fold_val_set:
        raise RuntimeError(f"Fold {fold_idx + 1}: pseudo/gold pool overlap or leakage.")

    n_gold = len(fold_gold_train_ids)
    train_global = np.asarray([uid_to_idx[u] for u in fold_train_ids], dtype=np.int64)
    gold_global = np.asarray([uid_to_idx[u] for u in fold_gold_train_ids], dtype=np.int64)
    pseudo_global = np.asarray([uid_to_idx[u] for u in pseudo_train_ids], dtype=np.int64)

    gold_observed = float(mask_matrix[gold_global].sum())
    pseudo_observed_per_study = mask_matrix[pseudo_global].sum(axis=1)
    avg_pseudo_observed = float(pseudo_observed_per_study.mean())
    if gold_observed <= 0 or avg_pseudo_observed <= 0:
        raise ValueError(f"Fold {fold_idx + 1}: no observed gold/pseudo labels.")

    pseudo_target_mass = PSEUDO_MASS_RATIO_5R * gold_observed
    n_pseudo_epoch = int(math.ceil(
        pseudo_target_mass / (PSEUDO_WEIGHT_5R * avg_pseudo_observed)
    ))
    n_pseudo_epoch = min(max(n_pseudo_epoch, 1), len(pseudo_train_ids))
    actual_pseudo_mass_ratio = (
        n_pseudo_epoch * PSEUDO_WEIGHT_5R * avg_pseudo_observed / gold_observed
    )

    train_dataset = FiveRoleStudyDataset(fold_train_ids)
    val_dataset = FiveRoleStudyDataset(fold_val_ids)
    sampler = FiveRoleGoldPrioritizedSampler(
        gold_positions=np.arange(n_gold, dtype=np.int64),
        pseudo_positions=np.arange(n_gold, len(fold_train_ids), dtype=np.int64),
        n_pseudo=n_pseudo_epoch,
        seed=SEED + fold_idx * 1000,
    )
    train_loader = DataLoader(
        train_dataset,
        batch_size=FIVE_ROLE_BATCH_SIZE,
        sampler=sampler,
        num_workers=FIVE_ROLE_NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=1,
        shuffle=False,
        num_workers=FIVE_ROLE_NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
    )

    train_masks = mask_matrix[train_global]
    train_labels = y_matrix[train_global]
    train_weights = np.where(
        source_names[train_global] == "gold", GOLD_WEIGHT_5R, PSEUDO_WEIGHT_5R
    ).astype(np.float32)
    weighted_mask = train_masks * train_weights[:, None]
    weighted_positive = (train_labels * weighted_mask).sum(axis=0)
    weighted_negative = ((1.0 - train_labels) * weighted_mask).sum(axis=0)
    pos_weight_np = np.clip(
        weighted_negative / np.maximum(weighted_positive, 1e-6), 1.0, 10.0
    ).astype(np.float32)
    pos_weight = torch.tensor(pos_weight_np, dtype=torch.float32, device=device)

    return {
        "fold_idx": fold_idx,
        "fold_number": fold_idx + 1,
        "gold_train_ids": fold_gold_train_ids,
        "val_ids": fold_val_ids,
        "train_ids": fold_train_ids,
        "train_dataset": train_dataset,
        "val_dataset": val_dataset,
        "train_sampler": sampler,
        "train_loader": train_loader,
        "val_loader": val_loader,
        "pos_weight": pos_weight,
        "pos_weight_np": pos_weight_np,
        "n_pseudo_epoch": n_pseudo_epoch,
        "actual_pseudo_mass_ratio": actual_pseudo_mass_ratio,
        "gold_observed": gold_observed,
        "avg_pseudo_observed": avg_pseudo_observed,
    }


# Materialize fold 1 for the existing Cell J smoke test and keep the legacy
# variable names available. Cell K will create each fold afresh in sequence.
fold0_data_5r = build_five_role_fold_data(0)
gold_train_ids = fold0_data_5r["gold_train_ids"]
val_study_ids = fold0_data_5r["val_ids"]
train_ids_5r = fold0_data_5r["train_ids"]
val_ids_5r = fold0_data_5r["val_ids"]
five_role_train_dataset = fold0_data_5r["train_dataset"]
five_role_val_dataset = fold0_data_5r["val_dataset"]
five_role_train_sampler = fold0_data_5r["train_sampler"]
five_role_train_loader = fold0_data_5r["train_loader"]
five_role_val_loader = fold0_data_5r["val_loader"]
pos_weight_5r = fold0_data_5r["pos_weight"]
n_pseudo_epoch_5r = fold0_data_5r["n_pseudo_epoch"]

print("========== FIVE-FOLD DATA PIPELINE READY ==========")
print("Gold studies:", len(gold_all_ids), "| folds:", CV_N_SPLITS)
print("Fold 1: gold train", len(gold_train_ids), "| validation", len(val_study_ids))
print("Pseudo pool per fold:", len(pseudo_train_ids))
print("Pseudo studies sampled per epoch (fold 1):", n_pseudo_epoch_5r)
print("Actual pseudo/gold weighted mass ratio (fold 1):", round(
    fold0_data_5r["actual_pseudo_mass_ratio"], 3
))
print("Fold 1 train batches:", len(five_role_train_loader))


## 17. Cell I — Five-role 2.5D DINOv2 architecture

In [ ]:
# ============================================================
# CELL I — Five-role 2.5D DINOv2 model (V2: target x role pooling)
# Shared ViT-S/14 encoder; each target pools each role independently;
# five role vectors are concatenated and sent to a target-specific head.
# ============================================================
import math
import torch
import torch.nn as nn


class FiveRoleDINOv2_2p5D(nn.Module):
    def __init__(
        self,
        backbone,
        image_size=IMAGE_SIZE,
        backbone_dim=BACKBONE_DIM,
        embed_dim=EMBED_DIM,
        freeze_early_blocks=FREEZE_EARLY_BLOCKS,
        dropout=0.15,
        encoder_chunk_size=8,
    ):
        super().__init__()
        self.backbone = backbone
        self.image_size = int(image_size)
        self.backbone_dim = int(backbone_dim)
        self.embed_dim = int(embed_dim)
        self.encoder_chunk_size = int(encoder_chunk_size)
        self.n_roles = len(FIVE_ROLE_ORDER)
        self.n_targets = len(TARGETS)
        self.n_windows = N_ROLE_WINDOWS

        if self.image_size % PATCH_SIZE != 0:
            raise ValueError("Image size must be divisible by DINOv2 patch size 14.")
        if not hasattr(self.backbone, "blocks") or len(self.backbone.blocks) != 12:
            raise ValueError("Expected DINOv2 ViT-S/14 with 12 transformer blocks.")
        if self.n_roles != 5:
            raise ValueError(f"Expected exactly five roles, found {self.n_roles}.")

        # Preserve the existing fine-tuning policy: freeze the backbone first,
        # then unfreeze its last blocks and final LayerNorm.
        for p in self.backbone.parameters():
            p.requires_grad_(False)
        if not 0 <= freeze_early_blocks < 12:
            raise ValueError("freeze_early_blocks must be between 0 and 11.")
        self.trainable_blocks = list(self.backbone.blocks[freeze_early_blocks:])
        for block in self.trainable_blocks:
            for p in block.parameters():
                p.requires_grad_(True)
        for p in self.backbone.norm.parameters():
            p.requires_grad_(True)

        # Convert each DINOv2 window into one compact feature vector.
        self.patch_attention = nn.Sequential(
            nn.LayerNorm(self.backbone_dim),
            nn.Linear(self.backbone_dim, 1, bias=False),
        )
        self.window_projection = nn.Sequential(
            nn.Linear(self.backbone_dim * 2, self.embed_dim),
            nn.LayerNorm(self.embed_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # Mild spatial prior; peripheral patches remain available to attention.
        grid = self.image_size // PATCH_SIZE
        coords = torch.linspace(-1.0, 1.0, grid)
        yy, xx = torch.meshgrid(coords, coords, indexing="ij")
        log_gaussian = -0.5 * ((xx / 0.55) ** 2 + (yy / 0.55) ** 2)
        log_gaussian = log_gaussian - log_gaussian.max()
        self.register_buffer("center_bias", (0.15 * log_gaussian.reshape(-1)).float())

        self.role_embedding = nn.Parameter(
            torch.zeros(1, self.n_roles, 1, self.embed_dim)
        )
        self.window_embedding = nn.Embedding(self.n_windows, self.embed_dim)
        self.position_projection = nn.Sequential(
            nn.Linear(3, self.embed_dim),
            nn.GELU(),
            nn.Linear(self.embed_dim, self.embed_dim),
        )
        nn.init.normal_(self.role_embedding, std=0.02)

        # V2: independent attention query for every (target, role) pair.
        # Shape: [12 targets, 5 roles, embedding dimension].
        self.target_role_queries = nn.Parameter(
            torch.empty(self.n_targets, self.n_roles, self.embed_dim)
        )
        nn.init.normal_(self.target_role_queries, std=0.5)

        # h(target, role) = global_role + gamma * (attended_role - global_role).
        # Start gamma at 0.70, not 0.10, so target-specific evidence is not
        # almost suppressed at initialization.
        gamma_init = 0.70
        gamma_logit = math.log(gamma_init / (1.0 - gamma_init))
        self.target_role_residual_logits = nn.Parameter(
            torch.full((self.n_targets, self.n_roles), gamma_logit)
        )
        self.role_feature_norm = nn.LayerNorm(self.embed_dim)

        # Every target gets its own prediction head. Each head sees the five
        # role-specific vectors concatenated in FIVE_ROLE_ORDER, plus presence
        # indicators so a missing role is distinguishable from a real zero.
        head_input_dim = self.n_roles * self.embed_dim + self.n_roles
        self.target_classifiers = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(head_input_dim),
                nn.Linear(head_input_dim, 128),
                nn.GELU(),
                nn.Dropout(0.25),
                nn.Linear(128, 1),
            )
            for _ in range(self.n_targets)
        ])

        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        self.register_buffer("image_mean", mean)
        self.register_buffer("image_std", std)

    def train(self, mode=True):
        super().train(mode)
        # Frozen DINO blocks remain in eval mode; selected final blocks train.
        self.backbone.eval()
        for block in self.trainable_blocks:
            block.train(mode)
        self.backbone.norm.train(mode)
        return self

    def forward(self, images, window_valid, window_positions):
        # images: [B, 5, 10, 3, 224, 224], uint8
        # masks/positions: [B, 5, 10]
        if images.ndim != 6:
            raise ValueError(f"Expected 6D images, received {tuple(images.shape)}")
        B, R, W, C, H, WW = images.shape
        expected = (self.n_roles, self.n_windows, 3, self.image_size, self.image_size)
        if (R, W, C, H, WW) != expected:
            raise ValueError(
                f"Unexpected image shape {tuple(images.shape)}; expected [B, {expected}]."
            )
        if window_valid.shape != (B, R, W):
            raise ValueError("window_valid shape does not match images.")
        if window_positions.shape != (B, R, W):
            raise ValueError("window_positions shape does not match images.")
        if not window_valid.bool().any(dim=(1, 2)).all():
            raise ValueError("At least one study in the batch has no valid windows.")

        # Encode only valid windows; do not waste DINOv2 compute on padded slots.
        flat_valid = window_valid.bool().reshape(-1)
        valid_indices = flat_valid.nonzero(as_tuple=False).squeeze(1)
        flat_images = images.reshape(B * R * W, C, H, WW)
        selected = flat_images.index_select(0, valid_indices).float().div_(255.0)
        selected = (selected - self.image_mean) / self.image_std

        encoded_chunks = []
        for start in range(0, selected.shape[0], self.encoder_chunk_size):
            chunk = selected[start:start + self.encoder_chunk_size]
            feat = self.backbone.forward_features(chunk)
            cls_token = feat["x_norm_clstoken"]
            patch_tokens = feat["x_norm_patchtokens"]
            patch_scores = self.patch_attention(patch_tokens).squeeze(-1)
            patch_scores = patch_scores + self.center_bias.unsqueeze(0)
            patch_weights = torch.softmax(patch_scores, dim=1)
            patch_pool = torch.sum(
                patch_tokens * patch_weights.unsqueeze(-1), dim=1
            )
            encoded_chunks.append(
                self.window_projection(torch.cat([cls_token, patch_pool], dim=-1))
            )
        valid_features = torch.cat(encoded_chunks, dim=0)

        flat_features = valid_features.new_zeros((B * R * W, self.embed_dim))
        flat_features = flat_features.index_copy(0, valid_indices, valid_features)
        tokens = flat_features.view(B, R, W, self.embed_dim)

        # Add role, window-order and normalized slice-position information.
        pos = window_positions.to(dtype=tokens.dtype).clamp(0.0, 1.0)
        pos_features = torch.stack(
            [pos, torch.sin(2.0 * math.pi * pos), torch.cos(2.0 * math.pi * pos)],
            dim=-1,
        )
        pos_features = self.position_projection(pos_features)
        order_ids = torch.arange(W, device=images.device)
        tokens = (
            tokens
            + self.role_embedding.to(dtype=tokens.dtype)
            + self.window_embedding(order_ids).view(1, 1, W, self.embed_dim)
            + pos_features
        )

        # Per-role global means: [B, R, D]. Missing roles remain zero.
        valid = window_valid.bool()
        mask_f = valid.unsqueeze(-1).to(tokens.dtype)
        role_counts = mask_f.sum(dim=2).clamp_min(1.0)
        role_global = (tokens * mask_f).sum(dim=2) / role_counts

        # Independent attention over windows WITHIN each role for each target.
        # Scores: [B, T, R, W]; masked softmax safely handles absent roles.
        scores = torch.einsum(
            "brwd,trd->btrw", tokens, self.target_role_queries
        ) / math.sqrt(self.embed_dim)
        scores = scores.masked_fill(
            ~valid[:, None, :, :], torch.finfo(scores.dtype).min
        )
        attention = torch.softmax(scores.float(), dim=-1).to(tokens.dtype)
        attention = attention * valid[:, None, :, :].to(tokens.dtype)
        attention = attention / attention.sum(dim=-1, keepdim=True).clamp_min(1e-6)
        attended_role = torch.einsum("btrw,brwd->btrd", attention, tokens)

        gamma = torch.sigmoid(self.target_role_residual_logits).view(
            1, self.n_targets, self.n_roles, 1
        )
        target_role_features = role_global[:, None, :, :] + gamma * (
            attended_role - role_global[:, None, :, :]
        )
        target_role_features = self.role_feature_norm(target_role_features)

        # Keep missing-role vectors at zero after LayerNorm; the presence flag
        # below explicitly tells the target head that the role was unavailable.
        role_presence = valid.any(dim=-1).to(target_role_features.dtype)  # [B, R]
        target_role_features = target_role_features * role_presence[:, None, :, None]

        # Concatenate five independently pooled role vectors per target.
        target_role_features = target_role_features.reshape(
            B, self.n_targets, self.n_roles * self.embed_dim
        )
        presence_features = role_presence[:, None, :].expand(
            -1, self.n_targets, -1
        )
        classifier_inputs = torch.cat(
            [target_role_features, presence_features], dim=-1
        )

        # Independent nonlinear prediction head for each target.
        logits = torch.cat(
            [
                self.target_classifiers[t](classifier_inputs[:, t, :])
                for t in range(self.n_targets)
            ],
            dim=1,
        )
        return logits


# ============================================================
# CV BACKBONE TEMPLATE — reload pristine official weights once.
# Each fold will receive a fresh deepcopy in Cell K.
# ============================================================
import copy
import gc

# Release model/optimizer objects from previous single-split experiments.
globals().pop("five_role_model", None)
globals().pop("five_role_backbone", None)
globals().pop("PRETRAINED_BACKBONE", None)
globals().pop("optimizer_5r", None)
globals().pop("scheduler_5r", None)
globals().pop("scaler_5r", None)
globals().pop("ckpt_5r", None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if "load_official_dinov2_backbone" not in globals():
    raise RuntimeError(
        "load_official_dinov2_backbone is not defined. Run the DINOv2 setup cell first."
    )

_cv_base_backbone, _cv_base_source = load_official_dinov2_backbone()
if not hasattr(_cv_base_backbone, "forward_features"):
    raise TypeError("Fresh pretrained backbone has no forward_features().")
if not hasattr(_cv_base_backbone, "blocks") or len(_cv_base_backbone.blocks) != 12:
    raise RuntimeError("Expected official DINOv2 ViT-S/14 with 12 blocks.")

PRETRAINED_BACKBONE_TEMPLATE = copy.deepcopy(_cv_base_backbone).cpu().eval()
del _cv_base_backbone
gc.collect()
print("Pristine pretrained backbone template:", _cv_base_source)

# Build one model only for Cell J smoke test. Cell K creates a fresh copy per fold.
five_role_backbone = copy.deepcopy(PRETRAINED_BACKBONE_TEMPLATE).to(device)
five_role_model = FiveRoleDINOv2_2p5D(
    backbone=five_role_backbone,
    freeze_early_blocks=FREEZE_EARLY_BLOCKS,
    encoder_chunk_size=8,
).to(device)

n_params_total_5r = sum(p.numel() for p in five_role_model.parameters())
n_params_trainable_5r = sum(
    p.numel() for p in five_role_model.parameters() if p.requires_grad
)
print("Five-role V2 model created for CV smoke test.")
print("Role order:", FIVE_ROLE_ORDER)
print("Targets:", len(TARGETS), "independent output heads")
print("Target-role queries:", tuple(five_role_model.target_role_queries.shape))
print("Forward method:", five_role_model.forward.__qualname__)
print("Total parameters:", round(n_params_total_5r / 1e6, 2), "M")
print("Trainable parameters:", round(n_params_trainable_5r / 1e6, 2), "M")


## 18. Cell J — Model forward + masked-loss smoke test (không train)

In [ ]:
# ============================================================
# CELL J — Five-role model smoke test; no optimizer or backward pass.
# ============================================================
from contextlib import nullcontext
import torch
import torch.nn.functional as F
import gc

smoke_5r = next(iter(five_role_val_loader))
smoke_images_5r = smoke_5r["images"].to(device, non_blocking=True)
smoke_valid_5r = smoke_5r["window_valid"].to(device, non_blocking=True)
smoke_positions_5r = smoke_5r["window_positions"].to(device, non_blocking=True)
smoke_labels_5r = smoke_5r["labels"].to(device, non_blocking=True)
smoke_masks_5r = smoke_5r["label_mask"].to(device, non_blocking=True)

five_role_model.eval()
with torch.inference_mode():
    amp_ctx = (
        torch.autocast(device_type="cuda", dtype=torch.float16)
        if device.type == "cuda" else nullcontext()
    )
    with amp_ctx:
        smoke_logits_5r = five_role_model(
            smoke_images_5r, smoke_valid_5r, smoke_positions_5r
        )
    smoke_loss_5r = F.binary_cross_entropy_with_logits(
        smoke_logits_5r.float(), smoke_labels_5r,
        reduction="none",
    )
    smoke_loss_5r = (
        smoke_loss_5r * smoke_masks_5r
    ).sum() / smoke_masks_5r.sum().clamp_min(1.0)

print("========== FIVE-ROLE MODEL SMOKE TEST ==========")
print("Images:", tuple(smoke_images_5r.shape))
print("Valid windows per role:", smoke_valid_5r[0].sum(dim=1).cpu().tolist())
print("Logits:", tuple(smoke_logits_5r.shape))
print("Finite logits:", bool(torch.isfinite(smoke_logits_5r).all().item()))
print("Masked validation-batch BCE (untrained weights):", float(smoke_loss_5r))
assert smoke_logits_5r.shape == (1, len(TARGETS))
assert torch.isfinite(smoke_logits_5r).all()
assert torch.isfinite(smoke_loss_5r)
print("PASS — five-role model, target pooling, availability masks and loss are executable.")
print("NOTE — this is an untrained model; this BCE is not a performance result.")

del smoke_5r, smoke_images_5r, smoke_valid_5r, smoke_positions_5r
# Keep no references to smoke outputs before training.
del smoke_labels_5r, smoke_masks_5r, smoke_logits_5r, smoke_loss_5r
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 19. Cell K — Training configuration, checkpoint/resume và training loop

**Mặc định không chạy training.** Sau khi Cell J báo `PASS`, đặt `RUN_FIVE_ROLE_TRAINING = True` trong cell này và chạy lại chính cell K. Batch size 1 + gradient accumulation 4 được giữ để tiết kiệm VRAM. Checkpoint được lưu trong `five_role_2p5d_v1/`, tách biệt với kết quả mô hình cũ.


In [ ]:
# ============================================================
# CELL K — Train five independent folds and write out-of-fold predictions
# ============================================================
import copy
import gc
import time
import json
import hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from pathlib import Path
from contextlib import nullcontext
from sklearn.metrics import roc_auc_score

FIVE_ROLE_CV_ROOT = OUTPUT_ROOT / "five_role_2p5d_v2_cv5_improved"
FIVE_ROLE_CV_ROOT.mkdir(parents=True, exist_ok=True)
FIVE_ROLE_CV_OOF_CSV = FIVE_ROLE_CV_ROOT / "oof_predictions.csv"
FIVE_ROLE_CV_FOLD_SUMMARY_CSV = FIVE_ROLE_CV_ROOT / "fold_summary.csv"

FIVE_ROLE_EPOCHS = 10
FIVE_ROLE_LR_BACKBONE = 2e-6
FIVE_ROLE_LR_HEAD = 1e-4
FIVE_ROLE_WEIGHT_DECAY = 0.03
FIVE_ROLE_PATIENCE = 3
FIVE_ROLE_MIN_DELTA = 1e-4
FIVE_ROLE_MAX_GRAD_NORM = 1.0
FIVE_ROLE_USE_AMP = torch.cuda.is_available()
FIVE_ROLE_RESUME_CV = True
RUN_FIVE_ROLE_CV = True  # Set True only after Cell J reports PASS.


def _five_role_amp_context():
    if device.type == "cuda" and FIVE_ROLE_USE_AMP:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


def five_role_loss(logits, labels, masks, sample_weights, pos_weight):
    per_cell = F.binary_cross_entropy_with_logits(
        logits.float(), labels.float(), reduction="none", pos_weight=pos_weight
    )
    weighted_mask = masks.float() * sample_weights.float().unsqueeze(1)
    # Preserve the established policy: pseudo-label loss is down-weighted by 0.05.
    denominator = masks.float().sum().clamp_min(1.0)
    return (per_cell * weighted_mask).sum() / denominator


@torch.no_grad()
def five_role_validate(model, loader):
    model.eval()
    total_loss_sum = 0.0
    total_observed = 0.0
    all_logits, all_labels, all_masks, all_uids = [], [], [], []
    for batch in loader:
        images = batch["images"].to(device, non_blocking=True)
        valid = batch["window_valid"].to(device, non_blocking=True)
        positions = batch["window_positions"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        masks = batch["label_mask"].to(device, non_blocking=True)
        with _five_role_amp_context():
            logits = model(images, valid, positions)
        per_cell = F.binary_cross_entropy_with_logits(
            logits.float(), labels.float(), reduction="none"
        )
        total_loss_sum += float((per_cell * masks).sum().item())
        total_observed += float(masks.sum().item())
        all_logits.append(logits.float().cpu().numpy())
        all_labels.append(labels.cpu().numpy())
        all_masks.append(masks.cpu().numpy())
        all_uids.extend(batch["uid"])
    if not all_logits:
        raise RuntimeError("Validation loader produced no batches.")
    return (
        total_loss_sum / max(total_observed, 1.0),
        np.concatenate(all_logits),
        np.concatenate(all_labels),
        np.concatenate(all_masks),
        all_uids,
    )


def five_role_train_one_epoch(model, loader, optimizer, scaler, sampler, pos_weight, epoch):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    sampler.set_epoch(epoch)
    loss_sum = 0.0
    n_batches = 0
    for step, batch in enumerate(loader, start=1):
        images = batch["images"].to(device, non_blocking=True)
        valid = batch["window_valid"].to(device, non_blocking=True)
        positions = batch["window_positions"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        masks = batch["label_mask"].to(device, non_blocking=True)
        sample_weights = batch["sample_weight"].to(device, non_blocking=True)

        group_start = ((step - 1) // FIVE_ROLE_GRAD_ACCUM) * FIVE_ROLE_GRAD_ACCUM + 1
        group_size = min(FIVE_ROLE_GRAD_ACCUM, len(loader) - group_start + 1)
        with _five_role_amp_context():
            logits = model(images, valid, positions)
            raw_loss = five_role_loss(
                logits, labels, masks, sample_weights, pos_weight
            )
            scaled_loss = raw_loss / group_size
        scaler.scale(scaled_loss).backward()

        do_step = (step % FIVE_ROLE_GRAD_ACCUM == 0) or (step == len(loader))
        if do_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                FIVE_ROLE_MAX_GRAD_NORM,
            )
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        loss_sum += float(raw_loss.detach().item())
        n_batches += 1
        if step % 100 == 0 or step == len(loader):
            print(
                f"step {step}/{len(loader)} | train_loss={loss_sum / max(n_batches, 1):.5f}",
                flush=True,
            )
    return loss_sum / max(n_batches, 1)


def _cpu_state_dict(model):
    return {key: value.detach().cpu() for key, value in model.state_dict().items()}


def _fold_signature(fold_data):
    payload = {
        "architecture": "DINOv2 ViT-S/14 V2 target-by-role attention + 12 heads",
        "fold": fold_data["fold_number"],
        "train_gold_sha256": hashlib.sha256(
            "\n".join(fold_data["gold_train_ids"]).encode()
        ).hexdigest(),
        "validation_sha256": hashlib.sha256(
            "\n".join(fold_data["val_ids"]).encode()
        ).hexdigest(),
        "pseudo_sha256": hashlib.sha256("\n".join(pseudo_train_ids).encode()).hexdigest(),
        "epochs": FIVE_ROLE_EPOCHS,
        "patience": FIVE_ROLE_PATIENCE,
        "lr_backbone": FIVE_ROLE_LR_BACKBONE,
        "lr_head": FIVE_ROLE_LR_HEAD,
        "weight_decay": FIVE_ROLE_WEIGHT_DECAY,
        "batch_size": FIVE_ROLE_BATCH_SIZE,
        "grad_accum": FIVE_ROLE_GRAD_ACCUM,
        "pseudo_weight": PSEUDO_WEIGHT_5R,
        "pseudo_mass_ratio": PSEUDO_MASS_RATIO_5R,
        "pos_weight": fold_data["pos_weight_np"].round(6).tolist(),
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:24]


def _fold_auc_summary(labels, masks, logits):
    probs = 1.0 / (1.0 + np.exp(-np.clip(logits, -30, 30)))
    aucs = []
    for target_idx in range(len(TARGETS)):
        valid = masks[:, target_idx] > 0
        y_true = labels[valid, target_idx].astype(int)
        if len(np.unique(y_true)) == 2:
            aucs.append(float(roc_auc_score(y_true, probs[valid, target_idx])))
    return float(np.mean(aucs)) if aucs else float("nan")


if not RUN_FIVE_ROLE_CV:
    print("5-fold training is disabled.")
    print("Cell J must report PASS first; then set RUN_FIVE_ROLE_CV=True and rerun Cell K.")
else:
    if "PRETRAINED_BACKBONE_TEMPLATE" not in globals():
        raise RuntimeError("Missing pristine pretrained backbone template. Rerun Cell I.")

    # Free the one-batch smoke-test model; each CV fold will create its own fresh model.
    for _name in ["five_role_model", "five_role_backbone"]:
        globals().pop(_name, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    all_fold_prediction_frames = []
    all_fold_summaries = []

    for fold_idx in range(CV_N_SPLITS):
        fold_data = build_five_role_fold_data(fold_idx)
        fold_number = fold_data["fold_number"]
        fold_dir = FIVE_ROLE_CV_ROOT / f"fold_{fold_number}"
        fold_dir.mkdir(parents=True, exist_ok=True)
        best_ckpt_path = fold_dir / "best_checkpoint.pt"
        last_ckpt_path = fold_dir / "last_checkpoint.pt"
        history_path = fold_dir / "training_history.csv"
        predictions_path = fold_dir / "oof_predictions_fold.csv"
        done_path = fold_dir / "fold_complete.json"
        signature = _fold_signature(fold_data)

        # Skip a fully completed fold only when its data/config signature matches.
        if done_path.is_file() and predictions_path.is_file():
            with open(done_path, "r", encoding="utf-8") as handle:
                done_meta = json.load(handle)
            if done_meta.get("signature") != signature:
                raise RuntimeError(
                    f"Fold {fold_number} was completed with a different configuration. "
                    "Use a new CV output directory rather than mixing experiments."
                )
            print(f"Fold {fold_number}/{CV_N_SPLITS}: already complete; reusing saved OOF predictions.")
            all_fold_prediction_frames.append(pd.read_csv(predictions_path))
            all_fold_summaries.append(done_meta["summary"])
            continue

        print("\n" + "=" * 18 + f" FOLD {fold_number}/{CV_N_SPLITS} " + "=" * 18)
        print("Gold train:", len(fold_data["gold_train_ids"]),
              "| gold validation:", len(fold_data["val_ids"]),
              "| pseudo pool:", len(pseudo_train_ids))

        # Every fold starts from the same pristine pretrained DINOv2 weights.
        fold_seed = SEED + fold_number * 100
        torch.manual_seed(fold_seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(fold_seed)
        np.random.seed(fold_seed)
        fold_backbone = copy.deepcopy(PRETRAINED_BACKBONE_TEMPLATE).to(device)
        fold_model = FiveRoleDINOv2_2p5D(
            backbone=fold_backbone,
            freeze_early_blocks=FREEZE_EARLY_BLOCKS,
            encoder_chunk_size=8,
        ).to(device)

        backbone_params = [p for p in fold_model.backbone.parameters() if p.requires_grad]
        backbone_param_ids = {id(p) for p in backbone_params}
        head_params = [
            p for p in fold_model.parameters()
            if p.requires_grad and id(p) not in backbone_param_ids
        ]
        if not backbone_params or not head_params:
            raise RuntimeError(f"Fold {fold_number}: missing trainable backbone/head parameters.")

        optimizer = torch.optim.AdamW(
            [
                {"params": backbone_params, "lr": FIVE_ROLE_LR_BACKBONE},
                {"params": head_params, "lr": FIVE_ROLE_LR_HEAD},
            ],
            weight_decay=FIVE_ROLE_WEIGHT_DECAY,
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=1, min_lr=1e-7
        )
        scaler = torch.cuda.amp.GradScaler(enabled=FIVE_ROLE_USE_AMP)

        start_epoch = 0
        best_val_loss = float("inf")
        best_epoch = -1
        early_stop_counter = 0
        history = []

        if FIVE_ROLE_RESUME_CV and last_ckpt_path.is_file():
            try:
                resume_ckpt = torch.load(
                    last_ckpt_path, map_location="cpu", weights_only=False
                )
            except TypeError:
                resume_ckpt = torch.load(last_ckpt_path, map_location="cpu")
            if resume_ckpt.get("signature") != signature:
                raise RuntimeError(
                    f"Fold {fold_number}: last checkpoint signature mismatch. "
                    "Do not resume an incompatible fold."
                )
            fold_model.load_state_dict(resume_ckpt["model_state"])
            optimizer.load_state_dict(resume_ckpt["optimizer_state"])
            scheduler.load_state_dict(resume_ckpt["scheduler_state"])
            if resume_ckpt.get("scaler_state"):
                scaler.load_state_dict(resume_ckpt["scaler_state"])
            start_epoch = int(resume_ckpt["epoch"])
            best_val_loss = float(resume_ckpt["best_val_loss"])
            best_epoch = int(resume_ckpt["best_epoch"])
            early_stop_counter = int(resume_ckpt["early_stop_counter"])
            history = resume_ckpt.get("history", [])
            print("Resuming fold at next epoch:", start_epoch + 1)
        else:
            print("Starting fold from pristine pretrained weights.")

        for epoch in range(start_epoch, FIVE_ROLE_EPOCHS):
            start_time = time.time()
            train_loss = five_role_train_one_epoch(
                fold_model,
                fold_data["train_loader"],
                optimizer,
                scaler,
                fold_data["train_sampler"],
                fold_data["pos_weight"],
                epoch,
            )
            val_loss, val_logits, val_labels, val_masks, val_uids = five_role_validate(
                fold_model, fold_data["val_loader"]
            )
            scheduler.step(val_loss)
            elapsed = time.time() - start_time
            val_macro_auc = _fold_auc_summary(val_labels, val_masks, val_logits)
            history.append({
                "fold": fold_number,
                "epoch": epoch + 1,
                "train_loss": train_loss,
                "gold_validation_bce": val_loss,
                "gold_validation_macro_auc": val_macro_auc,
                "lr_backbone": optimizer.param_groups[0]["lr"],
                "lr_head": optimizer.param_groups[1]["lr"],
                "minutes": elapsed / 60.0,
            })
            pd.DataFrame(history).to_csv(history_path, index=False)

            improved = val_loss < best_val_loss - FIVE_ROLE_MIN_DELTA
            if improved:
                best_val_loss = float(val_loss)
                best_epoch = epoch + 1
                early_stop_counter = 0
                torch.save({
                    "model_state": _cpu_state_dict(fold_model),
                    "targets": list(TARGETS),
                    "role_order": list(FIVE_ROLE_ORDER),
                    "fold": fold_number,
                    "best_epoch": best_epoch,
                    "best_val_loss": best_val_loss,
                    "signature": signature,
                }, best_ckpt_path)
            else:
                early_stop_counter += 1

            torch.save({
                "epoch": epoch + 1,
                "model_state": _cpu_state_dict(fold_model),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "scaler_state": scaler.state_dict(),
                "best_val_loss": best_val_loss,
                "best_epoch": best_epoch,
                "early_stop_counter": early_stop_counter,
                "history": history,
                "signature": signature,
            }, last_ckpt_path)

            print(
                f"Fold {fold_number} | epoch {epoch + 1}/{FIVE_ROLE_EPOCHS} "
                f"train={train_loss:.5f} val_bce={val_loss:.5f} "
                f"val_auc={val_macro_auc:.4f} best_epoch={best_epoch} "
                f"({elapsed / 60.0:.1f} min)",
                flush=True,
            )
            if early_stop_counter >= FIVE_ROLE_PATIENCE:
                print(f"Fold {fold_number}: early stopping triggered.")
                break

        if not best_ckpt_path.is_file():
            raise FileNotFoundError(f"Fold {fold_number}: no best checkpoint was saved.")
        try:
            best_fold_ckpt = torch.load(
                best_ckpt_path, map_location="cpu", weights_only=False
            )
        except TypeError:
            best_fold_ckpt = torch.load(best_ckpt_path, map_location="cpu")
        if best_fold_ckpt.get("signature") != signature:
            raise RuntimeError(f"Fold {fold_number}: best checkpoint signature mismatch.")
        fold_model.load_state_dict(best_fold_ckpt["model_state"])

        fold_val_loss, fold_logits, fold_labels, fold_masks, fold_uids = five_role_validate(
            fold_model, fold_data["val_loader"]
        )
        fold_probs = 1.0 / (1.0 + np.exp(-np.clip(fold_logits, -30, 30)))
        expected_val_uids = set(fold_data["val_ids"])
        if set(fold_uids) != expected_val_uids or len(fold_uids) != len(expected_val_uids):
            raise RuntimeError(f"Fold {fold_number}: validation UID coverage mismatch.")

        fold_prediction_rows = []
        for row_idx, uid in enumerate(fold_uids):
            for target_idx, target in enumerate(TARGETS):
                fold_prediction_rows.append({
                    "StudyInstanceUID": str(uid),
                    "fold": fold_number,
                    "target": target,
                    "y_true": int(fold_labels[row_idx, target_idx]),
                    "label_observed": bool(fold_masks[row_idx, target_idx] > 0),
                    "probability": float(fold_probs[row_idx, target_idx]),
                })
        fold_predictions_df = pd.DataFrame(fold_prediction_rows)
        fold_predictions_df.to_csv(predictions_path, index=False)

        fold_macro_auc = _fold_auc_summary(fold_labels, fold_masks, fold_logits)
        summary = {
            "fold": fold_number,
            "n_gold_train": len(fold_data["gold_train_ids"]),
            "n_gold_validation": len(fold_data["val_ids"]),
            "n_pseudo_pool": len(pseudo_train_ids),
            "n_pseudo_sampled_per_epoch": int(fold_data["n_pseudo_epoch"]),
            "best_epoch": int(best_fold_ckpt["best_epoch"]),
            "gold_validation_bce": float(fold_val_loss),
            "gold_validation_macro_auc": float(fold_macro_auc),
            "signature": signature,
        }
        with open(done_path, "w", encoding="utf-8") as handle:
            json.dump({"signature": signature, "summary": summary}, handle, indent=2)

        all_fold_prediction_frames.append(fold_predictions_df)
        all_fold_summaries.append(summary)
        print("Saved fold predictions:", predictions_path)

        # Free the fold model/optimizer before initializing the next backbone.
        del fold_model, fold_backbone, optimizer, scheduler, scaler, fold_data
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    oof_predictions_df = pd.concat(all_fold_prediction_frames, ignore_index=True)
    if oof_predictions_df.duplicated(["StudyInstanceUID", "target"]).any():
        raise RuntimeError("OOF output has duplicate StudyInstanceUID/target rows.")
    if set(oof_predictions_df["StudyInstanceUID"].astype(str)) != set(gold_all_ids):
        raise RuntimeError("OOF predictions do not cover exactly all 58 gold studies.")
    if len(oof_predictions_df) != len(gold_all_ids) * len(TARGETS):
        raise RuntimeError(
            f"Expected {len(gold_all_ids) * len(TARGETS)} OOF rows, "
            f"got {len(oof_predictions_df)}."
        )
    oof_predictions_df = oof_predictions_df.sort_values(
        ["StudyInstanceUID", "target"]
    ).reset_index(drop=True)
    oof_predictions_df.to_csv(FIVE_ROLE_CV_OOF_CSV, index=False)
    fold_summary_df = pd.DataFrame(all_fold_summaries).sort_values("fold")
    fold_summary_df.to_csv(FIVE_ROLE_CV_FOLD_SUMMARY_CSV, index=False)

    print("\n========== FIVE-FOLD CV TRAINING COMPLETE ==========")
    print("OOF gold studies:", oof_predictions_df["StudyInstanceUID"].nunique(), "/", len(gold_all_ids))
    print("OOF rows:", len(oof_predictions_df))
    print("OOF predictions:", FIVE_ROLE_CV_OOF_CSV)
    print("Fold summary:", FIVE_ROLE_CV_FOLD_SUMMARY_CSV)
    display(fold_summary_df)


## 20. Cell L — Gold validation metrics và predictions

In [ ]:
# ============================================================
# CELL L — Aggregate out-of-fold predictions across all 58 gold studies
# ============================================================
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

FIVE_ROLE_CV_ROOT = OUTPUT_ROOT / "five_role_2p5d_v2_cv5"
FIVE_ROLE_CV_OOF_CSV = FIVE_ROLE_CV_ROOT / "oof_predictions.csv"
FIVE_ROLE_CV_FOLD_SUMMARY_CSV = FIVE_ROLE_CV_ROOT / "fold_summary.csv"

if not FIVE_ROLE_CV_OOF_CSV.is_file():
    print("No complete OOF file found. Run Cell K with RUN_FIVE_ROLE_CV=True first.")
else:
    oof_df = pd.read_csv(FIVE_ROLE_CV_OOF_CSV)
    required = {
        "StudyInstanceUID", "fold", "target", "y_true",
        "label_observed", "probability",
    }
    missing = required - set(oof_df.columns)
    if missing:
        raise ValueError(f"OOF CSV is missing columns: {sorted(missing)}")
    if oof_df.duplicated(["StudyInstanceUID", "target"]).any():
        raise ValueError("Duplicate StudyInstanceUID/target rows in OOF predictions.")
    if set(oof_df["target"].unique()) != set(TARGETS):
        raise ValueError("OOF targets do not match TARGETS.")
    if oof_df["StudyInstanceUID"].nunique() != len(gold_all_ids):
        raise ValueError("OOF output does not cover all gold studies.")

    observed_flag = oof_df["label_observed"].astype(str).str.strip().str.lower().isin(
        {"true", "1", "1.0"}
    )
    metric_rows = []
    fold_auc_rows = []
    for target in TARGETS:
        target_df = oof_df[
            (oof_df["target"] == target) & observed_flag
        ].copy()
        y_true = target_df["y_true"].astype(int).to_numpy()
        y_prob = target_df["probability"].astype(float).to_numpy()
        auc_defined = len(np.unique(y_true)) == 2
        pooled_auc = float(roc_auc_score(y_true, y_prob)) if auc_defined else np.nan
        pooled_f1 = float(f1_score(y_true, y_prob >= 0.5, zero_division=0)) if len(y_true) else np.nan
        pooled_accuracy = float(accuracy_score(y_true, y_prob >= 0.5)) if len(y_true) else np.nan

        target_fold_aucs = []
        for fold_number in range(1, CV_N_SPLITS + 1):
            fold_df = target_df[target_df["fold"] == fold_number]
            fy = fold_df["y_true"].astype(int).to_numpy()
            fp = fold_df["probability"].astype(float).to_numpy()
            defined = len(np.unique(fy)) == 2
            fold_auc = float(roc_auc_score(fy, fp)) if defined else np.nan
            if defined:
                target_fold_aucs.append(fold_auc)
            fold_auc_rows.append({
                "target": target,
                "fold": fold_number,
                "n_observed": int(len(fy)),
                "n_positive": int(fy.sum()) if len(fy) else 0,
                "n_negative": int((1 - fy).sum()) if len(fy) else 0,
                "roc_auc": fold_auc,
            })

        metric_rows.append({
            "target": target,
            "n_observed_oof": int(len(y_true)),
            "n_positive_oof": int(y_true.sum()) if len(y_true) else 0,
            "n_negative_oof": int((1 - y_true).sum()) if len(y_true) else 0,
            "pooled_oof_roc_auc": pooled_auc,
            "mean_defined_fold_auc": float(np.mean(target_fold_aucs)) if target_fold_aucs else np.nan,
            "std_defined_fold_auc": float(np.std(target_fold_aucs, ddof=1)) if len(target_fold_aucs) > 1 else np.nan,
            "f1_at_0.5": pooled_f1,
            "accuracy_at_0.5": pooled_accuracy,
            "auc_defined": bool(auc_defined),
        })

    metrics_df = pd.DataFrame(metric_rows)
    fold_auc_df = pd.DataFrame(fold_auc_rows)
    defined_auc = metrics_df["pooled_oof_roc_auc"].dropna()

    observed_df = oof_df[observed_flag].copy()
    y = observed_df["y_true"].astype(float).to_numpy()
    p = np.clip(observed_df["probability"].astype(float).to_numpy(), 1e-7, 1 - 1e-7)
    pooled_oof_bce = float(-(y * np.log(p) + (1.0 - y) * np.log(1.0 - p)).mean())

    metrics_path = FIVE_ROLE_CV_ROOT / "oof_metrics.csv"
    fold_auc_path = FIVE_ROLE_CV_ROOT / "fold_target_auc.csv"
    metrics_df.to_csv(metrics_path, index=False)
    fold_auc_df.to_csv(fold_auc_path, index=False)

    print("========== FIVE-FOLD OUT-OF-FOLD GOLD EVALUATION ==========")
    print("Gold studies covered:", oof_df["StudyInstanceUID"].nunique(), "/", len(gold_all_ids))
    print("Observed target cells:", len(observed_df))
    print("Pooled OOF BCE:", round(pooled_oof_bce, 5))
    print(
        "Pooled OOF macro ROC-AUC:",
        round(float(defined_auc.mean()), 5) if len(defined_auc) else "N/A",
    )
    print("Targets with defined pooled OOF AUC:", len(defined_auc), "/", len(TARGETS))
    print("\nPer-target OOF metrics:")
    display(metrics_df)
    print("\nPer-fold target AUC (some folds may have one class and therefore NaN):")
    display(fold_auc_df)

    if FIVE_ROLE_CV_FOLD_SUMMARY_CSV.is_file():
        print("\nPer-fold training summary:")
        display(pd.read_csv(FIVE_ROLE_CV_FOLD_SUMMARY_CSV))
    print("Saved metrics:", metrics_path)
    print("Saved fold AUC:", fold_auc_path)
    print(
        "Interpretation note: pooled OOF AUC uses predictions from five separately trained "
        "models; also inspect per-fold AUC and its variability because fold-specific score "
        "calibration can differ. Best-epoch selection uses each fold's own validation BCE, "
        "so this is a practical CV estimate, not a fully nested unbiased test estimate."
    )


## Kết thúc và cách tiếp tục

- `five_role_2p5d_v1/best_checkpoint.pt`: checkpoint có gold-validation BCE tốt nhất.
- `five_role_2p5d_v1/last_checkpoint.pt`: trạng thái gần nhất để resume.
- `training_history.csv`, `validation_metrics.csv`, `validation_predictions.csv`: log và kết quả đánh giá.

Validation chỉ có 12 gold studies; macro AUC theo từng target có thể rất rời rạc và không ổn định. Không nên chọn mô hình chỉ dựa trên một thay đổi nhỏ của AUC này.


In [ ]:
# ============================================================
# ONE-TIME EXPORT — Five-role V2 CV5 submission bundle
# Run this once in Notebook 07 AFTER all five CV folds are complete.
# It does NOT train or modify any checkpoint.
# ============================================================
import json
import shutil
import zipfile
from pathlib import Path
import torch

CV_ROOT = OUTPUT_ROOT / "five_role_2p5d_v2_cv5_improved"
BUNDLE_PATH = Path("/kaggle/working/five_role_v2_cv5_submission_bundle.zip")
STAGING = Path("/kaggle/working/_five_role_v2_cv5_bundle_staging")

TARGETS_FOR_BUNDLE = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]
ROLE_ORDER_FOR_BUNDLE = [
    "SAG_FLUID", "COR_FLUID", "AX_FLUID", "SAG_NONFLUID", "COR_NONFLUID"
]

if not CV_ROOT.is_dir():
    raise FileNotFoundError(f"CV output directory not found: {CV_ROOT}")

# Locate the exact DINOv2 source repository already used by torch.hub in training.
hub_dir = Path(torch.hub.get_dir())
repo_candidates = [
    p for p in hub_dir.glob("facebookresearch_dinov2_*")
    if (p / "hubconf.py").is_file() and (p / "dinov2").is_dir()
]
if not repo_candidates:
    raise FileNotFoundError(
        f"Official DINOv2 Hub source not found under {hub_dir}. "
        "Run Cell 15 in Notebook 07 first so torch.hub has the official repository cached."
    )
if len(repo_candidates) > 1:
    print("DINOv2 source candidates:")
    for p in repo_candidates:
        print(" -", p)
DINOV2_REPO = sorted(repo_candidates, key=lambda p: p.stat().st_mtime)[-1]

# Verify that all five BEST checkpoints exist and belong to the expected fold/architecture.
fold_metadata = []
for fold in range(1, 6):
    checkpoint_path = CV_ROOT / f"fold_{fold}" / "best_checkpoint.pt"
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Missing fold {fold} best checkpoint: {checkpoint_path}")
    try:
        ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    except TypeError:
        ckpt = torch.load(checkpoint_path, map_location="cpu")
    if ckpt.get("fold") != fold:
        raise ValueError(f"Fold metadata mismatch in {checkpoint_path}: {ckpt.get('fold')}")
    if list(ckpt.get("targets", [])) != TARGETS_FOR_BUNDLE:
        raise ValueError(f"Target order mismatch in fold {fold} checkpoint.")
    if list(ckpt.get("role_order", [])) != ROLE_ORDER_FOR_BUNDLE:
        raise ValueError(f"Role order mismatch in fold {fold} checkpoint.")
    if "model_state" not in ckpt or not ckpt["model_state"]:
        raise ValueError(f"Fold {fold} checkpoint has no model_state.")
    fold_metadata.append({
        "fold": fold,
        "best_epoch": int(ckpt.get("best_epoch", -1)),
        "best_val_loss": float(ckpt.get("best_val_loss", float("nan"))),
        "signature": str(ckpt.get("signature", "")),
    })
    del ckpt

# Build a compact folder tree. Only BEST weights are included.
if STAGING.exists():
    shutil.rmtree(STAGING)
STAGING.mkdir(parents=True)

repo_dst = STAGING / "dinov2_repo"
shutil.copytree(
    DINOV2_REPO,
    repo_dst,
    ignore=shutil.ignore_patterns(
        ".git", "__pycache__", "*.pyc", "tests", "notebooks", ".github"
    ),
)

bundle_cv_root = STAGING / "five_role_2p5d_v2_cv5"
for fold in range(1, 6):
    dst_dir = bundle_cv_root / f"fold_{fold}"
    dst_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(
        CV_ROOT / f"fold_{fold}" / "best_checkpoint.pt",
        dst_dir / "best_checkpoint.pt",
    )

manifest = {
    "architecture": "DINOv2 ViT-S/14 shared 2.5D five-role; target-by-role attention; 12 independent heads",
    "backbone_source": "facebookresearch/dinov2 (local source snapshot from torch.hub cache)",
    "backbone_model": "dinov2_vits14",
    "backbone_pretrained_weights_included": False,
    "note": "Each best_checkpoint.pt contains the full fine-tuned model state, including the entire DINOv2 backbone. Submission instantiates the local architecture with pretrained=False and loads the complete checkpoint state.",
    "image_size": 224,
    "patch_size": 14,
    "backbone_dim": 384,
    "embed_dim": 256,
    "freeze_early_blocks": 9,
    "n_role_slices": 12,
    "n_role_windows": 10,
    "targets": TARGETS_FOR_BUNDLE,
    "role_order": ROLE_ORDER_FOR_BUNDLE,
    "folds": fold_metadata,
}
(STAGING / "bundle_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

if BUNDLE_PATH.exists():
    BUNDLE_PATH.unlink()
with zipfile.ZipFile(BUNDLE_PATH, "w", compression=zipfile.ZIP_STORED) as zf:
    for path in sorted(STAGING.rglob("*")):
        if path.is_file():
            zf.write(path, arcname=path.relative_to(STAGING).as_posix())

with zipfile.ZipFile(BUNDLE_PATH, "r") as zf:
    bad_member = zf.testzip()
    if bad_member is not None:
        raise RuntimeError(f"Bundle ZIP verification failed at: {bad_member}")

print("========== FIVE-ROLE SUBMISSION BUNDLE READY ==========")
print("CV checkpoint root:", CV_ROOT)
print("DINOv2 source:", DINOV2_REPO)
print("Fold best epochs:", [x["best_epoch"] for x in fold_metadata])
print("Bundle:", BUNDLE_PATH)
print("Size (GiB):", round(BUNDLE_PATH.stat().st_size / 1024**3, 3))
print("Included: dinov2_repo/ + five_role_2p5d_v2_cv5/fold_1..fold_5/best_checkpoint.pt + bundle_manifest.json")
print("Excluded: last_checkpoint.pt, optimizer states, OOF CSVs, train cache, supplementary image cache.")
